# 🦴 RSNA Knee Abnormality Detection — 5-Fold ConvNeXt-Tiny SOTA Inference
This inference notebook executes a **100% offline, zero-RAM leakage, multi-fold ensemble** of the trained ConvNeXt-Tiny 2.5D models:
* **Clinical Preprocessing**: Direct DICOM streaming, 130mm FOV physical crop, LPS normal vector coordinate sorting, 3.5mm depth equidistant triplets (identical to `rsna_knee_preprocess_dataset.ipynb`).
* **SOTA Architecture**: FPN Top-Down 2D Spatial Fusion (Stage 3 + Stage 4), Trainable GeM Spatial Pooling, Intra-Plane 3D Volumetric Depth Interaction, Hierarchical Dual-Pooling (Context + Sharp Focal Peak Slice with Linear Focal Gating).
* **Optimized Inductive Priors**: Warm-Start Gaussian Slot Priors with bounded residual drift; Cross-Plane Attention pruned to preserve view-specific locality; FiLM visual feature injection pruned to eliminate visual noise while preserving direct clinical logit bias (`meta_finding_bias`).
* **Zero-RAM Streaming Execution**: Generator batch streaming ($B=8$, $<100\text{ MB}$ peak RAM), all 5 fold checkpoints preloaded concurrently into GPU VRAM ($<450\text{ MB}$, $<3\%$ GPU).
* **Robust Test-Time Augmentations (TTA)**: 3-Pass Physical Depth Slice-Jittering + Vectorized GPU Laterality Normalization ('R' $\to$ 'L') + Column-Wise Percentile Rank Normalization across Hidden Test set.


In [1]:
# ── 1. Setup & Environment Verification (100% Offline Compatible) ─────────────
import os, sys, gc, glob, math, random, warnings
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
from scipy.stats import rankdata
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

# Native Kaggle Pre-installed Packages (No Internet / No pip required)
import pydicom
import timm
import cv2

warnings.filterwarnings("ignore")
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print(f"🎮 GPU Device: {torch.cuda.get_device_name(0)}")
print(f"🚀 PyTorch Version: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
print("✅ All dependencies imported natively from Kaggle environment without internet.")

🎮 GPU Device: Tesla T4
🚀 PyTorch Version: 2.10.0+cu128 | CUDA Available: True
✅ All dependencies imported natively from Kaggle environment without internet.


In [2]:
import os, sys
import torch

# =========================================================================
# 2. Global Configuration (CFG) — 21-Slice 224px 5-Fold Trained Pipeline
# =========================================================================
class CFG:
    seed                 = 42
    img_size             = 224          # ⚡ 224x224 Native resolution matching trained pipeline
    crop_mm              = 130.0        # 130mm Standardized physical knee joint FOV crop
    target_triplet_depth_mm = 3.5       # 3.5mm physical thickness per triplet step (~7mm total span)
    # ── 🌟 Anatomical Physical Depth Slice Bands (from rsna_knee_preprocess_dataset.ipynb)
    slice_bands          = {
        'Sagittal': (0.08, 0.92),
        'Coronal':  (0.18, 0.82),
        'Axial':    (0.10, 0.90)
    }
    num_slices_per_plane = 7            # 7 Sagittal + 7 Coronal + 7 Axial = 21 slice positions
    total_slices         = 21
    num_slices           = 21           # Total slices across 3 planes: 7 x 3
    dropout              = 0.35         # Finding attention head dropout (matches trained model)
    unfreeze_last        = 2            # Backbone unfreeze depth (trained ConvNeXt checkpoint config)
    use_plane_dropout    = False        # Whole-plane dropout (inactive during inference)
    plane_dropout_prob   = 0.20
    channels             = 3            # True 2.5D: 3 adjacent slices (i-1, i, i+1)
    
    # ── 🌟 Clinical Multi-Plane & Finding-Specific Attention Head ────────
    head_type            = "cross_plane_finding_specific"
    use_cross_plane      = False        # 🌟 Disabled: preserve view-specific locality for Hierarchical Head
    use_intra_plane_depth   = False        # 🛡️ Disabled: Matches trained checkpoints
    depth_interact_dropout  = 0.10         # Depth residual dropout
    use_meta_film        = False        # 🛡️ Disabled: Pruned 768-dim visual FiLM injection
    cross_plane_heads    = 4            # 4 Multi-Head Attention Channels
    cross_plane_dropout  = 0.15         # Cross-Plane Dropout
    use_plane_embedding  = False        # 🛡️ Disabled View-Aware Plane Embeddings (matches training)
    use_cosine_classifier = True        # 🌟 Hyperspherical Cosine Metric Head
    use_multi_prototype  = False        # Single-Prototype Cosine Head (K=1)
    num_prototypes       = 1
    
    # ── 🌟 Dual-Scale Context Smoothing & Plane-Decoupled Hierarchical Architecture
    use_dual_pooling            = True  # Dual-Pooling: Attentive Context + Sharp Focal Peak Slice
    use_hierarchical_pooling    = True  # 🌟 Decouples Intra-Plane Slices from Inter-Plane Weighting
    use_plane_decoupled_pooling = True  # Checkpoint compatibility alias
    broad_temperature           = 2.0   # 🌟 Dual-Scale Context: soft smoothed temperature capturing broad joint context
    focal_temperature           = 2.0   # Alias for backwards compatibility

    # ── 🌊 Trainable GeM Spatial Pooling ─────────────────────────────────
    use_gem_pooling      = True         # Generalized Mean Pooling on ConvNeXt feature maps
    use_trainable_gem    = True         # Alias
    gem_p                = 3.0          # Spatial pooling power (matches checkpoint)
    gem_p_trainable      = True

    # ── ⚡ 2D Top-Down Feature Pyramid Network (FPN) & Multi-Stage Fusion ──
    use_fpn_fusion          = True      # 🌟 SOTA 2D Top-Down Channel-Gated FPN Spatial Fusion (Stage 3 [14x14] + Stage 4 [7x7])
    fpn_gamma_init          = 0.05       # FPN projection gate init
    use_multi_stage_fusion  = False     # 🛡️ Disabled 1D Stage3 Vector Fusion (matches training)
    use_stage3_fusion       = False     # Alias
    stage3_feat_dim         = 384       # ConvNeXt-Tiny Stage 3 channels
    stage3_gamma_init       = 0.05      # Initial gate value
    stage3_fusion_init      = 0.05      # Alias

    # ── 🩺 Bilateral Finding Relation Head ────────────────────────────────
    use_finding_relation = False        # 🌟 Disabled to match exp training
    use_bilateral_relation = False      # Alias
    relation_num_heads   = 4            # 4 Attention heads across 12 pathologies
    relation_gamma_init  = 0.05         # Finding relation gate init
    relation_dropout     = 0.10

    # ── 🧠 Warm-Start Residual Anatomical Gaussian Slot Prior ─────────────
    use_slot_prior       = True         # 🌟 Continuous Gaussian Inductive Prior
    slot_prior_strength  = 0.55         # Peak focal boost (+0.55 logit)
    slot_prior_sigma     = 0.90         # 1D Gaussian kernel standard deviation
    
    # ── 🧬 13-Dim Tabular Scanner & Patient Metadata Injection ───────────
    use_metadata_injection = False      # 🌟 Disabled Tabular Metadata (Pure Image Learning to match trained models)
    meta_dim               = 13         # 13 numerical/categorical DICOM features
    meta_logit_bias_weight = 0.06       # Prior logit bias weight (synchronized to 0.06)
    meta_mlp_dropout       = 0.20
    
    # ── 🌟 Auxiliary Multi-Plane Contrast Classification Task ────────────
    use_contrast_aux        = True      # Enabled during training
    use_contrast_embedding  = False
    n_contrast_types        = 4         # 4 contrast classes: 0=OTHER, 1=FS, 2=T1, 3=NOFS_FLUID
    use_dynamic_contrast_gating = False # 🌟 Contrast-Aware Dynamic Inter-Plane Gating (Default False, auto-detected)
    dynamic_plane_gamma         = 1.0   # Multiplier for contrast-driven plane gating logits
    
    # ── 🚀 Inference Hardware & Acceleration ──────────────────────────────
    batch_size           = 8            # Stream 8 studies concurrently (< 100MB peak RAM)
    num_workers          = 2            # Parallel CPU DICOM decompression threads
    backbone_chunk_size  = 672          # Slices chunking for backbone forward
    use_amp              = True         # FP16 Automatic Mixed Precision
    use_channels_last    = False        # Contiguous format ensures zero CUDA stride / illegal memory errors
    device               = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    
    # ── 🔄 Test-Time Augmentation (TTA) Suite ─────────────────────────────
    use_laterality_flip     = True      # Base Right 'R' -> Left 'L' geometric normalization
    use_slice_jitter_tta    = True      # 3-Pass Slice Jittering TTA (Standard, Forward, Backward)
    use_anatomical_flip_tta = False     # 🛡️ Disabled: Purely use 3-Pass Slice Jitter TTA
    use_rank_norm           = True      # 🌟 Post-Processing: Percentile Rank Normalization
    
    # ── 📁 Competition & Model Checkpoint Search Paths ────────────────────
    comp_dir             = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
    if not os.path.exists(comp_dir) and os.path.exists("./rsna-knee-abnormality-detection"):
        comp_dir = "./rsna-knee-abnormality-detection"
        
    backbone             = "convnext_tiny.in12k_ft_in1k"
    ckpt_candidates      = [
        "/kaggle/input/datasets/mediumguy/rsna-convnext-tiny-2-stage",
        ".",
        "/kaggle/input/rsna-convnext-exp",
        "/kaggle/input/rsna-convnext-cur-baseline",
        "/kaggle/input/rsna-convnext-baseline",
        "/kaggle/input/rsna-convnext-baseline-old-focal-gate",
        "/kaggle/input/rsna-convnext-kaggle-models",
        "/kaggle/input/rsna-convnext-tiny",
        "/kaggle/input/datasets/mediumguy/rsna-convnext-tiny",
        "/kaggle/input/rsna-trained-models",
        "/kaggle/input/rsna-checkpoints",
        "/kaggle/input/rsna-knee-models",
        "/kaggle/input/rsna-convnext-checkpoints",
        "/kaggle/input/rsna-convnext-models",
        "/kaggle/input/rsna-models"
    ]

TARGET_COLS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus',
    'Medial OA', 'Lateral OA', 'PF OA', 'Effusion',
    'Synovitis', "Baker's", 'Contusion', 'Fracture'
]

print(f"⚡ Device: {CFG.device} | Resolution: {CFG.img_size}x{CFG.img_size} | Slices: {CFG.num_slices} (3 planes x {CFG.num_slices_per_plane} slices)")
print(f"🧠 Prior: Gaussian Slot Prior (σ={CFG.slot_prior_sigma}) | Dual-Scale Context: {CFG.use_dual_pooling} (Broad τ={CFG.broad_temperature})")
print(f"🌟 Hierarchical Pooling: {CFG.use_hierarchical_pooling} | Depth Interaction: {CFG.use_intra_plane_depth} | FPN 2D Fusion: {CFG.use_fpn_fusion} | GeM: {CFG.use_gem_pooling} (p={CFG.gem_p})")
print(f"🩺 Finding Relation Head: {CFG.use_finding_relation} (Heads={CFG.relation_num_heads}) | ChannelsLast: {CFG.use_channels_last}")
print(f"🧬 Metadata Injection: Dim={CFG.meta_dim}, LogitBiasWeight={CFG.meta_logit_bias_weight} | CrossPlane={CFG.use_cross_plane} | MetaFiLM={getattr(CFG, 'use_meta_film', False)}")
print(f"🔄 TTA Suite: Slice-Jitter TTA={CFG.use_slice_jitter_tta} | Anatomical Flip TTA={CFG.use_anatomical_flip_tta} | RankNorm={CFG.use_rank_norm}")


⚡ Device: cuda:0 | Resolution: 224x224 | Slices: 21 (3 planes x 7 slices)
🧠 Prior: Gaussian Slot Prior (σ=0.9) | Dual-Scale Context: True (Broad τ=2.0)
🌟 Hierarchical Pooling: True | Depth Interaction: False | FPN 2D Fusion: True | GeM: True (p=3.0)
🩺 Finding Relation Head: False (Heads=4) | ChannelsLast: False
🧬 Metadata Injection: Dim=13, LogitBiasWeight=0.06 | CrossPlane=False | MetaFiLM=False
🔄 TTA Suite: Slice-Jitter TTA=True | Anatomical Flip TTA=False | RankNorm=True


In [3]:
# =========================================================================
# 3. Load Competition Data & Build Exact Series Mapping
# =========================================================================

sub_path = os.path.join(CFG.comp_dir, "sample_submission.csv")
sub_df = pd.read_csv(sub_path)
print(f"✅ Loaded Sample Submission ({len(sub_df):,} test cases) from: {sub_path}")

# Load test_series.csv directly from competition directory
test_series_path = os.path.join(CFG.comp_dir, "test_series.csv")
if not os.path.exists(test_series_path):
    # Fallback to train_series.csv if doing local/train dry run
    test_series_path = os.path.join(CFG.comp_dir, "train_series.csv")

test_series_df = pd.read_csv(test_series_path)
test_series_df["StudyInstanceUID"] = test_series_df["StudyInstanceUID"].astype(str)
test_series_df["SeriesInstanceUID"] = test_series_df["SeriesInstanceUID"].astype(str)

# 🛡️ Clean Fluid_Sensitive & Fat_Suppression flags: convert NaNs, strings, floats safely to 0 or 1
if 'Fluid_Sensitive' in test_series_df.columns:
    test_series_df['Fluid_Sensitive'] = pd.to_numeric(test_series_df['Fluid_Sensitive'], errors='coerce').fillna(0).astype(int)
else:
    test_series_df['Fluid_Sensitive'] = 0

if 'Fat_Suppression' in test_series_df.columns:
    test_series_df['Fat_Suppression'] = pd.to_numeric(test_series_df['Fat_Suppression'], errors='coerce').fillna(0).astype(int)
else:
    test_series_df['Fat_Suppression'] = 0

print(f"✅ Loaded Test Series records: {test_series_path} ({len(test_series_df):,} rows)")

# ── 4-Way Contrast Classification (Strict Match with rsna_knee_preprocess_dataset.ipynb) ──
CONTRAST_MAP = {"OTHER": 0, "FS": 1, "T1": 2, "NOFS_FLUID": 3, "UNK": 0}

def classify_series_contrast(row):
    """
    Classifies series contrast from Fluid_Sensitive, Fat_Suppression flags and optional SeriesDescription.
    Uses regex word boundaries (\b) to strictly avoid matching 'fse' (Fast Spin Echo) as 'fs'.
    Correctly separates Fat-Suppression (FS), T1, NOFS_FLUID, and OTHER.
    """
    import re
    fluid_flag = int(row.get("Fluid_Sensitive", 0) == 1)
    fat_flag   = int(row.get("Fat_Suppression", 0) == 1)
    desc       = str(row.get("SeriesDescription", "")).lower() if "SeriesDescription" in row else ""
    
    # 1. If SeriesDescription is present, parse with strict word boundaries
    if desc:
        has_fs    = bool(re.search(r"\b(fs|fat[_\s\-]?sat|fatsat|spair|stir|water)\b", desc))
        has_t1    = bool(re.search(r"\bt1\b", desc))
        has_fluid = bool(re.search(r"\b(pd|dp|proton|t2)\b", desc))
        
        if has_fs:
            return "FS"
        if has_t1 and not has_fs:
            return "T1"
        if has_fluid and not has_fs:
            return "NOFS_FLUID"
    
    # 2. Official Kaggle Ground Truth series flags
    if fat_flag == 1:
        return "FS"
    elif fluid_flag == 1 and fat_flag == 0:
        return "NOFS_FLUID"
    
    return "OTHER"

test_series_df["contrast"] = test_series_df.apply(classify_series_contrast, axis=1)
test_series_df["contrast_int"] = test_series_df["contrast"].map(CONTRAST_MAP).fillna(0).astype(int)

# ── Multi-Plane Series Selection (Strict 100% Match with rsna_knee_preprocess_dataset.ipynb) ──
def count_series_slices(study_id, series_id, base_dir=CFG.comp_dir):
    """Counts real DICOM files in series folder. Returns -1 if directory is missing."""
    for sdir in [
        os.path.join(base_dir, 'test_images', str(study_id), str(series_id)),
        os.path.join(base_dir, 'test_series', str(study_id), str(series_id)),
        os.path.join(base_dir, 'train_images', str(study_id), str(series_id)),
        os.path.join(base_dir, 'train_series', str(study_id), str(series_id)),
        os.path.join(base_dir, str(study_id), str(series_id))
    ]:
        if os.path.exists(sdir):
            return len([f for f in os.listdir(sdir) if f.lower().endswith('.dcm')])
    return -1

def inspect_series_header(study_id, series_id, base_dir=CFG.comp_dir):
    """
    Rapidly reads first DICOM file header (stop_before_pixels=True, <0.5ms):
    Returns (is_scout, is_suboptimal, is_pd, is_t2):
    - is_scout: True if scout, loc, survey -> MUST BE ABSOLUTELY REJECTED (never accepted)
    - is_suboptimal: True if repeat, redo, motion, corrupt -> used only as fallback if no clean series
    - is_pd: True if Proton Density (clinical gold standard for knee meniscus/ligaments)
    - is_t2: True if T2 sequence
    """
    for sdir in [
        os.path.join(base_dir, 'test_images', str(study_id), str(series_id)),
        os.path.join(base_dir, 'test_series', str(study_id), str(series_id)),
        os.path.join(base_dir, 'train_images', str(study_id), str(series_id)),
        os.path.join(base_dir, 'train_series', str(study_id), str(series_id)),
        os.path.join(base_dir, str(study_id), str(series_id))
    ]:
        if os.path.exists(sdir):
            dcms = [f for f in os.listdir(sdir) if f.lower().endswith('.dcm')]
            if dcms:
                try:
                    hdr = pydicom.dcmread(os.path.join(sdir, dcms[0]), stop_before_pixels=True)
                    desc = (str(getattr(hdr, 'SeriesDescription', '')) + ' ' + 
                            str(getattr(hdr, 'ProtocolName', ''))).lower()
                    is_scout = any(k in desc for k in ['scout', 'loc', 'survey'])
                    is_suboptimal = any(k in desc for k in ['repeat', 'redo', 'motion', 'corrupt'])
                    is_pd = any(k in desc for k in ['pd', 'dp', 'proton'])
                    is_t2 = ('t2' in desc) and not is_pd
                    return is_scout, is_suboptimal, is_pd, is_t2
                except Exception:
                    pass
                return False, False, False, False
    return False, False, False, False

def get_study_series_info_map(series_df, base_dir=CFG.comp_dir):
    """
    Picks the optimal series for each plane via clinical safety rules:
    1. Rejects sequences with < 7 slices on disk.
    2. ABSOLUTE REJECTION OF SCOUTS: Series identified as scout/localizer/survey are NEVER accepted.
    3. SUBOPTIMAL SERIES HANDLING: Prefers clean series; only falls back to repeat/motion if no clean series exists,
       and explicitly flags 'is_suboptimal = 1' in metadata for downstream tracking.
    4. Clinical Sequence Hierarchy: Clean (10000) > FS (1000) > Proton Density (100) > T2 (20) + n_slices.
    5. Deterministic Tie-Breaker on SeriesInstanceUID string guarantees 100% stable selection.
    """
    study_map = {}
    MIN_SLICES = CFG.num_slices_per_plane  # 7 distinct slices required
    
    for study_id, group in series_df.groupby('StudyInstanceUID'):
        planes_info = {}
        for plane in ['Sagittal', 'Coronal', 'Axial']:
            sub = group[group['Anatomical_Plane'] == plane].copy()
            if len(sub) == 0:
                planes_info[plane] = {'series_id': None, 'is_fs': 0, 'is_suboptimal': 0, 'contrast': 'UNK', 'contrast_int': 0, 'n_slices': 0}
                continue
            
            # Count physical slice files
            sub['n_slices'] = sub['SeriesInstanceUID'].apply(lambda sid: count_series_slices(study_id, sid, base_dir))
            
            # Filter candidates with at least MIN_SLICES on disk
            sub = sub[sub['n_slices'] >= MIN_SLICES].copy()
            if len(sub) == 0:
                planes_info[plane] = {'series_id': None, 'is_fs': 0, 'is_suboptimal': 0, 'contrast': 'UNK', 'contrast_int': 0, 'n_slices': 0}
                continue
            
            # Inspect DICOM header
            header_stats = sub['SeriesInstanceUID'].apply(lambda sid: inspect_series_header(study_id, sid, base_dir))
            sub['is_scout']      = [h[0] for h in header_stats]
            sub['is_suboptimal'] = [h[1] for h in header_stats]
            sub['is_pd']         = [h[2] for h in header_stats]
            sub['is_t2']         = [h[3] for h in header_stats]
            
            # 🛡️ ABSOLUTE HARD FILTER: Scouts are NEVER clinical diagnostic planes -> reject entirely
            sub = sub[~sub['is_scout']].copy()
            if len(sub) == 0:
                planes_info[plane] = {'series_id': None, 'is_fs': 0, 'is_suboptimal': 0, 'contrast': 'UNK', 'contrast_int': 0, 'n_slices': 0}
                continue
            
            # 🛡️ CLEAN VS SUBOPTIMAL: Prefer clean series. Only fallback to suboptimal (repeat/motion) if no clean exists
            clean_sub = sub[~sub['is_suboptimal']].copy()
            valid_sub = clean_sub if len(clean_sub) > 0 else sub.copy()
            
            # Clinical Hierarchical Priority:
            # Clean (+10000) > FS (+1000) > PD (+100) > T2 (+20) + slice_score
            valid_sub['slice_score'] = valid_sub['n_slices'].apply(lambda n: max(0, n))
            valid_sub['priority'] = (
                (~valid_sub['is_suboptimal']).astype(int) * 10000 +
                (valid_sub['contrast'] == 'FS').astype(int) * 1000 +
                valid_sub['is_pd'].astype(int) * 100 +
                valid_sub['is_t2'].astype(int) * 20 +
                valid_sub['slice_score']
            )
            
            # Deterministic tie-break
            valid_sub['sid_str'] = valid_sub['SeriesInstanceUID'].astype(str)
            valid_sub = valid_sub.sort_values(
                by=['priority', 'slice_score', 'sid_str'],
                ascending=[False, False, True]
            )
            best_row = valid_sub.iloc[0]
            
            c_str = str(best_row.get('contrast', 'UNK'))
            planes_info[plane] = {
                'series_id': str(best_row['SeriesInstanceUID']),
                'is_fs': int(c_str == 'FS'),
                'is_suboptimal': int(best_row.get('is_suboptimal', False)),
                'contrast': c_str,
                'contrast_int': CONTRAST_MAP.get(c_str, 0),
                'n_slices': int(best_row['slice_score'])
            }
        study_map[study_id] = planes_info
    return study_map

study_plane_map = get_study_series_info_map(test_series_df, CFG.comp_dir)
print(f"✅ Mapped Multi-Plane Series & Contrast types for {len(study_plane_map):,} Studies.")


✅ Loaded Sample Submission (3 test cases) from: /kaggle/input/competitions/rsna-knee-abnormality-detection/sample_submission.csv
✅ Loaded Test Series records: /kaggle/input/competitions/rsna-knee-abnormality-detection/test_series.csv (15 rows)
✅ Mapped Multi-Plane Series & Contrast types for 3 Studies.


In [4]:
# =========================================================================
# 4. Standardized 130mm FOV Cropping, DICOM Physics & 21-Slice Preprocessing
# =========================================================================

# ⚡ GPU Normalization Constants (0.450 / 0.225 Uniform Grayscale MRI)
IMAGENET_MEAN = torch.tensor([0.450, 0.450, 0.450]).view(1, 1, 3, 1, 1)
IMAGENET_STD  = torch.tensor([0.225, 0.225, 0.225]).view(1, 1, 3, 1, 1)

def determine_laterality(ds):
    """
    3-Tier Laterality Determination (Strict match with rsna_knee_preprocess_dataset.ipynb):
    1. Standard DICOM tag (ImageLaterality / Laterality)
    2. 3D Patient LPS Space Center (c_x < -15mm -> 'R', c_x > +15mm -> 'L')
    3. SeriesDescription / ProtocolName text scanning
    """
    # Tier 1: Standard tag
    lat = getattr(ds, 'ImageLaterality', None) or getattr(ds, 'Laterality', None)
    if lat in ['R', 'L', 'RIGHT', 'LEFT']:
        return 'R' if lat in ['R', 'RIGHT'] else 'L'

    # Tier 2: 3D LPS Space Image Center
    if hasattr(ds, 'ImagePositionPatient') and hasattr(ds, 'ImageOrientationPatient') and hasattr(ds, 'PixelSpacing'):
        try:
            ipp = np.array(ds.ImagePositionPatient, dtype=float)
            iop = np.array(ds.ImageOrientationPatient, dtype=float)
            ps  = np.array(ds.PixelSpacing, dtype=float)
            r, c = getattr(ds, 'Rows', 512), getattr(ds, 'Columns', 512)
            c_x = ipp[0] + (c / 2.0) * ps[1] * iop[0] + (r / 2.0) * ps[0] * iop[3]
            if c_x < -15.0: return 'R'
            if c_x > +15.0: return 'L'
        except Exception:
            pass

    # Tier 3: Text scanning fallback (Strict word-boundary matching matching rsna_knee_preprocess_dataset.ipynb)
    desc = (str(getattr(ds, 'SeriesDescription', '')) + ' ' + str(getattr(ds, 'ProtocolName', ''))).upper()
    if any(k in desc for k in [' RT', 'RIGHT', ' R ', 'R_KNEE', 'RT_KNEE']): return 'R'
    if any(k in desc for k in [' LT', 'LEFT', ' L ', 'L_KNEE', 'LT_KNEE']):  return 'L'

    return 'Unknown'

def read_dicom_slice(filepath):
    """
    Reads DICOM slice with RescaleSlope/Intercept, MONOCHROME1 inversion, 
    and returns raw array along with 3D physical position p and unit normal n
    (Strict match with rsna_knee_preprocess_dataset.ipynb).
    """
    ds = pydicom.dcmread(filepath, force=True)
    arr = ds.pixel_array.astype(np.float32)

    # 1. Apply RescaleSlope & RescaleIntercept
    slope = float(getattr(ds, 'RescaleSlope', 1.0) or 1.0)
    intercept = float(getattr(ds, 'RescaleIntercept', 0.0) or 0.0)
    if slope != 1.0 or intercept != 0.0:
        arr = arr * slope + intercept

    # 2. Invert MONOCHROME1 (High values dark -> invert)
    photometric = str(getattr(ds, 'PhotometricInterpretation', '')).upper()
    if photometric == 'MONOCHROME1':
        arr = arr.max() - arr

    # 3. Extract 3D physical coordinate p and unit normal vector n
    p = np.zeros(3, dtype=float)
    n = np.zeros(3, dtype=float)
    if hasattr(ds, 'ImagePositionPatient') and hasattr(ds, 'ImageOrientationPatient'):
        try:
            p = np.array(ds.ImagePositionPatient, dtype=float)
            iop = np.array(ds.ImageOrientationPatient, dtype=float)
            raw_n = np.cross(iop[:3], iop[3:])
            norm_n = np.linalg.norm(raw_n)
            n = (raw_n / norm_n) if norm_n > 1e-6 else np.array([0.0, 0.0, 1.0])
        except Exception:
            inst = float(getattr(ds, 'InstanceNumber', 0.0) or 0.0)
            p = np.array([0.0, 0.0, inst])
            n = np.array([0.0, 0.0, 1.0])
    else:
        inst = float(getattr(ds, 'InstanceNumber', 0.0) or 0.0)
        p = np.array([0.0, 0.0, inst])
        n = np.array([0.0, 0.0, 1.0])

    # 3-Tier Laterality Determination
    lat_str = determine_laterality(ds)

    # Pixel Spacing
    ps = getattr(ds, 'PixelSpacing', [0.45, 0.45])
    try:
        ps_row, ps_col = float(ps[0]), float(ps[1])
    except Exception:
        ps_row, ps_col = 0.45, 0.45

    # 🛡️ FIX: Whitelist check to prevent 'FEMALE' or 'MISSING' from falsely becoming 'M'
    raw_sex = str(getattr(ds, 'PatientSex', '')).strip().upper()
    if raw_sex in {'M', 'MALE'}:
        sex_str = 'M'
    elif raw_sex in {'F', 'FEMALE'}:
        sex_str = 'F'
    else:
        sex_str = 'Unknown'

    # MagneticFieldStrength (3.0T vs 1.5T detection)
    field_strength = float(getattr(ds, 'MagneticFieldStrength', 1.5) or 1.5)

    return arr, p, n, lat_str, (ps_row, ps_col), sex_str, field_strength

def compute_130mm_crop_box(H, W, ps_row=0.45, ps_col=0.45, crop_mm=CFG.crop_mm):
    """
    Calculates padding and centered bounding box for a standard 130mm physical crop.
    Ensures output region is strictly (ch, cw) with aspect ratio 1:1, WITHOUT DISTORTION.
    (Strict 100% match with rsna_knee_preprocess_dataset.ipynb).
    """
    ps_row = max(0.1, ps_row) if ps_row > 0 else 0.45
    ps_col = max(0.1, ps_col) if ps_col > 0 else 0.45
    
    ch = max(1, int(round(crop_mm / ps_row)))
    cw = max(1, int(round(crop_mm / ps_col)))
    
    # Symmetric padding if original image is smaller than target crop
    pad_t = max(0, (ch - H) // 2)
    pad_b = max(0, ch - H - pad_t)
    pad_l = max(0, (cw - W) // 2)
    pad_r = max(0, cw - W - pad_l)
    
    # Centered crop coordinates in the padded coordinate space (eliminates aspect ratio distortion!)
    new_H = H + pad_t + pad_b
    new_W = W + pad_l + pad_r
    
    y1 = max(0, (new_H - ch) // 2)
    y2 = y1 + ch
    x1 = max(0, (new_W - cw) // 2)
    x2 = x1 + cw
    
    return (y1, y2, x1, x2), (pad_t, pad_b, pad_l, pad_r)

def window_crop_and_resize_slice(slice_raw, box, pads, p1, p99_5, target_size=CFG.img_size):
    """
    Applies volume-level windowing -> symmetric padding if FOV < 130mm -> exact centered 130mm crop -> resize.
    Strictly preserves 1:1 physical aspect ratio without vertical/horizontal stretching.
    (Strict 100% match with rsna_knee_preprocess_dataset.ipynb).
    """
    # 1. Per-Stack (Volume-Level) Windowing
    if p99_5 > p1:
        norm = np.clip((slice_raw - p1) / (p99_5 - p1 + 1e-6), 0.0, 1.0) * 255.0
    else:
        norm = np.zeros_like(slice_raw)
    norm = norm.astype(np.uint8)
    
    # 2. Symmetric padding if any dimension < 130mm
    (pad_t, pad_b, pad_l, pad_r) = pads
    if pad_t > 0 or pad_b > 0 or pad_l > 0 or pad_r > 0:
        norm = cv2.copyMakeBorder(norm, pad_t, pad_b, pad_l, pad_r, cv2.BORDER_REFLECT)
        
    # 3. Exact 130mm Physical Crop in padded space
    (y1, y2, x1, x2) = box
    cropped = norm[y1:y2, x1:x2]
        
    # 4. Square resize to target_size (224x224)
    if cropped.shape[0] != target_size or cropped.shape[1] != target_size:
        cropped = cv2.resize(cropped, (target_size, target_size), interpolation=cv2.INTER_AREA)
        
    return cropped

def extract_triplets_optimized(vol_stack, k_arr, crop_box, pads, p1, p99_5, target_size=CFG.img_size, n_slices=CFG.num_slices_per_plane, extract_tta=CFG.use_slice_jitter_tta, band_lo=0.15, band_hi=0.85):
    """
    Optimized slice-cached True 2.5D triplet extractor with Physics-Driven 3.5mm depth.
    Samples centers evenly across PHYSICAL DEPTH (mm), not array indices.
    Caches unique slices so that 3-pass slice jittering TTA runs in 1x I/O time.
    (Strict 100% match with rsna_knee_preprocess_dataset.ipynb).
    """
    total_s = len(vol_stack)
    
    # 🌟 Samples centers evenly spaced across PHYSICAL DEPTH (mm), matching training
    k_min, k_max = float(k_arr[0]), float(k_arr[-1])
    k_span = max(1e-3, k_max - k_min)
    k_start = k_min + k_span * band_lo
    k_end   = k_min + k_span * band_hi
    target_k_locs = np.linspace(k_start, k_end, n_slices)
    
    # Map physical target coordinates to nearest available slice indices
    center_idxs = [int(np.argmin(np.abs(k_arr - tk))) for tk in target_k_locs]
    
    # Guaranteed 100% Unique Slice Sampling
    if len(set(center_idxs)) < n_slices:
        unique_set = sorted(list(set(center_idxs)))
        available = [i for i in range(total_s) if i not in unique_set]
        while len(unique_set) < n_slices and available:
            unique_set.append(available.pop(0))
        center_idxs = sorted(unique_set[:n_slices])

    pass_index_lists = [center_idxs]
    if extract_tta:
        fwd_idxs = [min(total_s - 1, i + 1) for i in center_idxs]
        bwd_idxs = [max(0, i - 1) for i in center_idxs]
        pass_index_lists.extend([fwd_idxs, bwd_idxs])

    target_delta = getattr(CFG, "target_triplet_depth_mm", 3.5)
    needed_slice_idxs = set()
    triplet_plan_per_pass = []

    for idx_list in pass_index_lists:
        pass_plan = []
        for c_i in idx_list:
            k_curr = k_arr[c_i]
            prev_i = int(np.argmin(np.abs(k_arr - (k_curr - target_delta))))
            next_i = int(np.argmin(np.abs(k_arr - (k_curr + target_delta))))
            
            # 🛡️ Guard Rule: Prevent duplicate identical slice collapse on BOTH boundaries (Fixed!)
            if prev_i == c_i:
                prev_i = c_i - 1 if c_i > 0 else min(total_s - 1, c_i + 1)
            if next_i == c_i:
                next_i = c_i + 1 if c_i < total_s - 1 else max(0, c_i - 1)

            pass_plan.append((prev_i, c_i, next_i))
            needed_slice_idxs.add(prev_i)
            needed_slice_idxs.add(c_i)
            needed_slice_idxs.add(next_i)
        triplet_plan_per_pass.append(pass_plan)

    slice_cache = {}
    for s_i in needed_slice_idxs:
        slice_cache[s_i] = window_crop_and_resize_slice(vol_stack[s_i], crop_box, pads, p1, p99_5, target_size)

    passes_triplets = []
    for pass_plan in triplet_plan_per_pass:
        triplets = []
        for (p_i, c_i, n_i) in pass_plan:
            ch_prev = slice_cache[p_i]
            ch_curr = slice_cache[c_i]
            ch_next = slice_cache[n_i]
            triplets.append(np.stack([ch_prev, ch_curr, ch_next], axis=0))
        while len(triplets) < n_slices:
            triplets.append(triplets[-1] if triplets else np.zeros((3, target_size, target_size), dtype=np.uint8))
        passes_triplets.append(np.stack(triplets[:n_slices], axis=0))

    return tuple(passes_triplets)

def extract_triplets_from_dicom_series(study_id, series_id, plane_name, base_dir=CFG.comp_dir, target_size=CFG.img_size, n_slices=CFG.num_slices_per_plane, extract_tta=CFG.use_slice_jitter_tta):
    if series_id is None:
        return None, "Unknown", "Unknown", 0, 0.45, 0.45, 0, 1.5
        
    series_dirs = [
        os.path.join(base_dir, "test_series", str(study_id), str(series_id)),
        os.path.join(base_dir, "test_images", str(study_id), str(series_id)),
        os.path.join(base_dir, "train_series", str(study_id), str(series_id)),
        os.path.join(base_dir, "train_images", str(study_id), str(series_id)),
        os.path.join(base_dir, str(study_id), str(series_id))
    ]
    target_dir = None
    for sdir in series_dirs:
        if os.path.exists(sdir):
            target_dir = sdir
            break
    if target_dir is None:
        return None, "Unknown", "Unknown", 0, 0.45, 0.45, 0, 1.5
    dcm_files = glob.glob(os.path.join(target_dir, "*.dcm"))
    if not dcm_files:
        return None, "Unknown", "Unknown", 0, 0.45, 0.45, 0, 1.5
        
    slices_data = []
    series_lat = "Unknown"
    series_sex = "Unknown"
    mfs_list = []
    for f in dcm_files:
        try:
            arr, p, n, lat, ps, sex, mfs = read_dicom_slice(f)
            if series_lat == "Unknown" and lat != "Unknown": series_lat = lat
            if series_sex == "Unknown" and sex != "Unknown": series_sex = sex
            mfs_list.append(mfs)
            slices_data.append({'p': p, 'n': n, 'arr': arr, 'ps': ps})
        except Exception:
            pass
    if not slices_data:
        return None, "Unknown", "Unknown", 0, 0.45, 0.45, 0, 1.5
        
    total_files = len(slices_data)
    if total_files < n_slices:
        return None, series_lat, series_sex, total_files, 0.45, 0.45, 0, 1.5
        
    # 🌟 NORMAL VECTOR SIGN NORMALIZATION & GLOBAL REFERENCE AXIS
    # Project all slices onto a SINGLE shared series reference unit normal vector n_ref
    # Strictly eliminates scanner-dependent 180° normal inversion!
    plane_axis_map = {'Sagittal': 0, 'Coronal': 1, 'Axial': 2}
    target_axis = plane_axis_map.get(plane_name, 0)
    
    med_n = np.median([item['n'] for item in slices_data], axis=0)
    norm_val = float(np.linalg.norm(med_n))
    unit_med_n = (med_n / norm_val) if norm_val > 1e-6 else np.array([1.0, 0.0, 0.0])
    sign_factor = 1.0 if unit_med_n[target_axis] >= 0 else -1.0
    n_ref = sign_factor * unit_med_n  # Global standardized reference unit normal
    
    for item in slices_data:
        item['k'] = float(np.dot(item['p'], n_ref))
        
    # Strictly sort along standardized 3D physical normal axis
    slices_data.sort(key=lambda x: x['k'])
    k_arr = np.array([item['k'] for item in slices_data], dtype=float)
    total_files = len(slices_data)
    
    # 🛡️ PHYSICAL DEPTH DIVERSITY GATE (tol = 0.01 mm = 10 um):
    # Accurately merges identical dual-echo or duplicate PACS slices while strictly preserving 3D thin-slices (>= 0.5 mm)
    # (Strict 100% match with rsna_knee_preprocess_dataset.ipynb)
    DEPTH_TOL_MM = 0.01
    unique_k_coords = []
    for k_val in k_arr:
        if not any(abs(k_val - uk) < DEPTH_TOL_MM for uk in unique_k_coords):
            unique_k_coords.append(k_val)
            
    distinct_depths = len(unique_k_coords)
    if distinct_depths < n_slices:
        # Rejects series that physically does not cover at least 7 distinct cross-sections
        # Returns None so that this plane is gracefully replaced by blank_plane (zeros), strictly matching training!
        return None, series_lat, series_sex, distinct_depths, 0.45, 0.45, 0, 1.5
    
    vol_stack = np.stack([item['arr'] for item in slices_data], axis=0)
    
    ps_rows = [item['ps'][0] for item in slices_data]
    ps_cols = [item['ps'][1] for item in slices_data]
    med_ps_row = float(np.median(ps_rows)) if ps_rows else 0.45
    med_ps_col = float(np.median(ps_cols)) if ps_cols else 0.45
    H, W = vol_stack.shape[1], vol_stack.shape[2]
    crop_box, pads = compute_130mm_crop_box(H, W, med_ps_row, med_ps_col, CFG.crop_mm)
    has_padding = int(pads[0] > 0 or pads[1] > 0 or pads[2] > 0 or pads[3] > 0)
    
    # Coordinate-Aligned 130mm Windowing directly on raw unpadded knee tissue
    ch_raw = max(1, int(round(CFG.crop_mm / med_ps_row)))
    cw_raw = max(1, int(round(CFG.crop_mm / med_ps_col)))
    y1_raw = max(0, (H - ch_raw) // 2)
    y2_raw = min(H, y1_raw + ch_raw)
    x1_raw = max(0, (W - cw_raw) // 2)
    x2_raw = min(W, x1_raw + cw_raw)
    cropped_raw = vol_stack[:, y1_raw:y2_raw, x1_raw:x2_raw] if (y2_raw > y1_raw and x2_raw > x1_raw) else vol_stack
    p1, p99_5 = np.percentile(cropped_raw, [1.0, 99.5])
    
    band_lo, band_hi = CFG.slice_bands.get(plane_name, (0.15, 0.85))
    passes_triplets = extract_triplets_optimized(vol_stack, k_arr, crop_box, pads, p1, p99_5, target_size, n_slices, extract_tta, band_lo, band_hi)
    
    med_mfs = float(np.median(mfs_list)) if mfs_list else 1.5
    return passes_triplets, series_lat, series_sex, distinct_depths, med_ps_row, med_ps_col, has_padding, med_mfs

def get_first_valid(vals, default=0.45):
    """Scans across Sagittal -> Coronal -> Axial metadata with safe fallback."""
    for v in vals:
        if v is not None and v > 0:
            return float(v)
    return float(default)

def process_study(study_id, plane_map, base_dir=CFG.comp_dir, target_size=CFG.img_size, extract_tta=CFG.use_slice_jitter_tta):
    """
    Processes a single study into 21 True 2.5D Triplets (21, 3, 224, 224) and 13-Dim Tabular Metadata.
    Strictly matches the metadata formulas, normal vector orientation, and series selection in training.
    """
    p_dict = plane_map.get(study_id, {})
    sag_info = p_dict.get("Sagittal", {})
    cor_info = p_dict.get("Coronal", {})
    axi_info = p_dict.get("Axial", {})
    
    sag_id = sag_info.get("series_id")
    cor_id = cor_info.get("series_id")
    axi_id = axi_info.get("series_id")
    n_s = CFG.num_slices_per_plane
    
    sag_res, lat_s, sex_s, s_cnt_s, ps_r_s, ps_c_s, pad_s, mfs_s = extract_triplets_from_dicom_series(study_id, sag_id, "Sagittal", base_dir, target_size, n_s, extract_tta)
    cor_res, lat_c, sex_c, s_cnt_c, ps_r_c, ps_c_c, pad_c, mfs_c = extract_triplets_from_dicom_series(study_id, cor_id, "Coronal",  base_dir, target_size, n_s, extract_tta)
    axi_res, lat_a, sex_a, s_cnt_a, ps_r_a, ps_c_a, pad_a, mfs_a = extract_triplets_from_dicom_series(study_id, axi_id, "Axial",    base_dir, target_size, n_s, extract_tta)
    
    blank_plane = np.zeros((n_s, 3, target_size, target_size), dtype=np.uint8)
    # 🛡️ Always create exactly 3 passes (or 1 if no TTA) across ALL studies to prevent batch IndexError
    n_passes = 3 if extract_tta else 1
    full_passes = []
    for pass_idx in range(n_passes):
        sag = sag_res[pass_idx] if (sag_res is not None and len(sag_res) > pass_idx and sag_res[pass_idx] is not None) else blank_plane
        cor = cor_res[pass_idx] if (cor_res is not None and len(cor_res) > pass_idx and cor_res[pass_idx] is not None) else blank_plane
        axi = axi_res[pass_idx] if (axi_res is not None and len(axi_res) > pass_idx and axi_res[pass_idx] is not None) else blank_plane
        full_21 = np.concatenate([sag, cor, axi], axis=0)  # (21, 3, 224, 224)
        full_passes.append(full_21)
    
    # Laterality & Patient Sex hierarchy matching rsna_knee_preprocess_dataset.ipynb
    study_lat = lat_c if lat_c != "Unknown" else (lat_a if lat_a != "Unknown" else lat_s)
    study_sex = sex_c if sex_c != "Unknown" else (sex_s if sex_s != "Unknown" else sex_a)
    
    # Contrast Flags: Sagittal, Coronal, Axial x 7 positions = 21 flags
    sag_c = sag_info.get("contrast_int", 0)
    cor_c = cor_info.get("contrast_int", 0)
    axi_c = axi_info.get("contrast_int", 0)
    contrast_flags = [sag_c]*n_s + [cor_c]*n_s + [axi_c]*n_s
    
    # 🌟 9-Dim Tabular Metadata Vector (3-Tier Fallback via get_first_valid)
    sex_m = 1.0 if study_sex == "M" else 0.0
    sex_f = 1.0 if study_sex == "F" else 0.0
    
    # 3-Tier Fallback scanning Sag -> Cor -> Axi -> 0.45
    ps_r_val = get_first_valid([ps_r_s, ps_r_c, ps_r_a], 0.45)
    ps_c_val = get_first_valid([ps_c_s, ps_c_c, ps_c_a], 0.45)
    ps_r = float(np.clip((ps_r_val - 0.45) / 0.15, -3.0, 3.0))
    ps_c = float(np.clip((ps_c_val - 0.45) / 0.15, -3.0, 3.0))
    cp   = 1.0 if (pad_s or pad_c or pad_a) else 0.0
    # 🛡️ Match training: missing plane has s_cnt=0 -> (0 - 25)/10 = -2.5 -> clipped to -2.0
    s_sag = float(np.clip((s_cnt_s - 25.0) / 10.0, -2.0, 2.0))
    s_cor = float(np.clip((s_cnt_c - 25.0) / 10.0, -2.0, 2.0))
    s_axi = float(np.clip((s_cnt_a - 25.0) / 10.0, -2.0, 2.0))
    is_r  = 1.0 if study_lat == "R" else 0.0
    
    # 🌟 Scanner 3.0T Magnetic Field Strength (1.0 if >= 2.5T, else 0.0)
    mfs_val = get_first_valid([mfs_s, mfs_c, mfs_a], 1.5)
    is_3t   = 1.0 if mfs_val >= 2.5 else 0.0
    
    # 🌟 Suboptimal Plane Flags from DICOM Header Series Info (Strict match with training inspect_series_header)
    s_sub = float(sag_info.get("is_suboptimal", 0)) if sag_res is not None else 0.0
    c_sub = float(cor_info.get("is_suboptimal", 0)) if cor_res is not None else 0.0
    a_sub = float(axi_info.get("is_suboptimal", 0)) if axi_res is not None else 0.0
    
    meta_vec = np.array([sex_m, sex_f, ps_r, ps_c, cp, s_sag, s_cor, s_axi, is_r, is_3t, s_sub, c_sub, a_sub], dtype=np.float32)
    
    return full_passes, study_lat, contrast_flags, meta_vec

print("✅ 21-Slice Preprocessing Engine ready with 3D Normal Sorter, 3-Tier Laterality, Physics 3.5mm Triplets & Distortion-Free 130mm Cropping.")


✅ 21-Slice Preprocessing Engine ready with 3D Normal Sorter, 3-Tier Laterality, Physics 3.5mm Triplets & Distortion-Free 130mm Cropping.


In [5]:
class WarmStartResidualGaussianSlotPrior(nn.Module):
    """
    🧠 Warm-Start Residual Anatomical Gaussian Slot Prior with Dedicated Geometry Learning.
    - Anchors: Initialized at verified clinical landmarks (mu_0, sigma_0, pi_0).
    - Bounded Residual Shifts: delta_mu, delta_sigma, delta_gate clamped via tanh.
    - Plane Gating: Learns which orthogonal plane is dominant for each pathology.
    """
    def __init__(self, n_classes=12, n_slices_per_plane=7):
        super().__init__()
        self.n_classes = n_classes
        self.n_slices_per_plane = n_slices_per_plane
        self.total_slices = n_slices_per_plane * 3

        # Normalized anchors mu_0, sigma_0, pi_0 for (12 classes, 3 planes: 0=Sagittal, 1=Coronal, 2=Axial)
        # Slices 0..6 normalized as s/6.0 in [0, 1]
        # Normalized anchors mu_0, sigma_0, pi_0 for (12 classes, 3 planes: 0=Sagittal, 1=Coronal, 2=Axial)
        # Slices 0..6 normalized as s/6.0 in [0, 1]
        # Anatomical alignment:
        # Sagittal: Medial (low index ~1.5) <-> Lateral (high index ~4.5) on normalized Left Knee standard.
        # Coronal: Anterior (front index ~1.5) <-> Posterior (back index ~4.5).
        # Axial: Anterior (patellofemoral ~1.5) <-> Posterior (popliteal/Baker ~4.8).
        anchors_mu = [
            [3.0/6.0, 3.0/6.0, 3.0/6.0], # ACL: Sagittal center, Coronal center
            [1.2/6.0, 3.5/6.0, 3.0/6.0], # MCL: Medial side in Sagittal, joint center in Coronal
            [1.5/6.0, 3.8/6.0, 3.0/6.0], # Medial Meniscus: Medial in Sagittal, central/posterior horn in Coronal
            [4.5/6.0, 3.8/6.0, 3.0/6.0], # Lateral Meniscus: Lateral in Sagittal, central/posterior horn in Coronal
            [1.5/6.0, 3.5/6.0, 3.0/6.0], # Medial OA: Medial in Sagittal, joint line in Coronal
            [4.5/6.0, 3.5/6.0, 3.0/6.0], # Lateral OA: Lateral in Sagittal, joint line in Coronal
            [3.0/6.0, 1.5/6.0, 1.5/6.0], # PF OA: Anterior in Coronal & Axial (patellofemoral joint)
            [3.0/6.0, 2.0/6.0, 1.5/6.0], # Effusion: Suprapatellar pouch (Anterior in Coronal & Axial)
            [3.0/6.0, 2.5/6.0, 2.0/6.0], # Synovitis: Anterior/central joint capsule
            [1.8/6.0, 4.8/6.0, 4.8/6.0], # Baker's: Posteromedial gastrocnemius-semimembranosus bursa (Posterior Coronal & Axial)
            [3.0/6.0, 3.0/6.0, 3.0/6.0], # Contusion: Multi-plane joint center
            [3.0/6.0, 3.0/6.0, 3.0/6.0], # Fracture: Multi-plane joint center
        ]

        anchors_sigma = [
            [0.25, 0.30, 0.30], # ACL
            [0.30, 0.25, 0.30], # MCL
            [0.20, 0.25, 0.30], # Medial Meniscus
            [0.20, 0.25, 0.30], # Lateral Meniscus
            [0.22, 0.25, 0.30], # Medial OA
            [0.25, 0.22, 0.30], # Lateral OA
            [0.25, 0.30, 0.25], # PF OA
            [0.28, 0.35, 0.28], # Effusion
            [0.28, 0.32, 0.28], # Synovitis
            [0.20, 0.30, 0.25], # Baker's
            [0.32, 0.30, 0.30], # Contusion
            [0.28, 0.26, 0.28], # Fracture
        ]

        anchors_pi = [
            [0.85, 0.15, 0.40], # ACL: Sagittal primary
            [0.15, 0.85, 0.40], # MCL: Coronal primary
            [0.85, 0.60, 0.25], # Medial Meniscus: Sagittal primary, Coronal secondary
            [0.85, 0.80, 0.25], # Lateral Meniscus: Sagittal primary, Coronal secondary
            [0.75, 0.90, 0.30], # Medial OA: Coronal primary, Sagittal secondary
            [0.80, 0.75, 0.30], # Lateral OA: Coronal & Sagittal
            [0.70, 0.30, 0.85], # PF OA: Axial & Sagittal
            [0.85, 0.40, 0.70], # Effusion: Sagittal & Axial
            [0.55, 0.80, 0.65], # Synovitis: Coronal & Axial
            [0.25, 0.15, 0.90], # Baker's: Axial primary
            [0.70, 0.75, 0.65], # Contusion: Multi-plane
            [0.70, 0.80, 0.65], # Fracture: Multi-plane
        ]

        self.register_buffer('mu_0', torch.tensor(anchors_mu, dtype=torch.float32))
        self.register_buffer('sigma_0', torch.tensor(anchors_sigma, dtype=torch.float32))
        self.register_buffer('pi_0', torch.tensor(anchors_pi, dtype=torch.float32))

        # Learnable residual parameters initialized to 0
        self.delta_mu    = nn.Parameter(torch.zeros(n_classes, 3, dtype=torch.float32))
        self.delta_sigma = nn.Parameter(torch.zeros(n_classes, 3, dtype=torch.float32))
        self.delta_gate  = nn.Parameter(torch.zeros(n_classes, 3, dtype=torch.float32))

        s_grid = torch.linspace(0.0, 1.0, n_slices_per_plane)
        self.register_buffer('s_grid', s_grid)

    def get_geometry(self):
        mu = torch.clamp(self.mu_0 + 0.25 * torch.tanh(self.delta_mu), 0.05, 0.95)
        sigma = torch.clamp(self.sigma_0 * torch.exp(0.5 * torch.tanh(self.delta_sigma)), 0.08, 1.0)
        pi = torch.clamp(self.pi_0 + 0.30 * torch.tanh(self.delta_gate), 0.05, 0.98)
        return mu, sigma, pi

    def forward(self):
        mu, sigma, pi = self.get_geometry()
        s = self.s_grid.view(1, 1, self.n_slices_per_plane)
        m = mu.unsqueeze(-1)
        sig = sigma.unsqueeze(-1)
        p_val = pi.unsqueeze(-1) * torch.exp(-0.5 * ((s - m) / sig) ** 2)
        p_matrix = p_val.permute(1, 2, 0).reshape(self.total_slices, self.n_classes)
        return p_matrix

    def print_learned_anatomical_centers(self):
        mu, sigma, pi = self.get_geometry()
        mu = mu.detach().cpu().numpy()
        sigma = sigma.detach().cpu().numpy()
        pi = pi.detach().cpu().numpy()
        mu_0 = self.mu_0.cpu().numpy()

        scale_s = float(self.n_slices_per_plane - 1)
        print("\n" + "="*96)
        print("🧠 WARM-START RESIDUAL ANATOMICAL CENTERS & ADAPTIVE LESION GEOMETRY")
        print("="*96)
        print(f"{'Target Pathology':<18} | {'Sagittal (Center [Drift] / Spread / Gate)':<41} | {'Coronal (Center [Drift] / Spread / Gate)'}")
        print("-" * 96)

        for i, c_name in enumerate(TARGET_COLS):
            s_c = mu[i, 0] * scale_s
            s_d = (mu[i, 0] - mu_0[i, 0]) * scale_s
            s_sp = sigma[i, 0] * scale_s
            s_g = pi[i, 0] * 100.0
            s_sign = "+" if s_d >= 0 else ""
            s_str = f"L{s_c:.1f} [{s_sign}{s_d:.1f}sl] (±{s_sp:.1f}sl, {s_g:.0f}%)"

            c_c = mu[i, 1] * scale_s
            c_d = (mu[i, 1] - mu_0[i, 1]) * scale_s
            c_sp = sigma[i, 1] * scale_s
            c_g = pi[i, 1] * 100.0
            c_sign = "+" if c_d >= 0 else ""
            c_str = f"L{c_c:.1f} [{c_sign}{c_d:.1f}sl] (±{c_sp:.1f}sl, {c_g:.0f}%)"

            a_c = mu[i, 2] * scale_s
            a_d = (mu[i, 2] - mu_0[i, 2]) * scale_s
            a_sp = sigma[i, 2] * scale_s
            a_g = pi[i, 2] * 100.0
            a_sign = "+" if a_d >= 0 else ""
            a_str = f"Axial: L{a_c:.1f} [{a_sign}{a_d:.1f}sl] (±{a_sp:.1f}sl, {a_g:.0f}%)"

            print(f"{c_name:<18} | {s_str:<41} | {c_str}")
            print(f"{'':<18} | {a_str}")
            print("-" * 96)


import timm

class GeMPool2d(nn.Module):
    """
    🌊 Generalized Mean (GeM) Pooling (Radenović et al., TPAMI 2018):
    f = (1 / |Omega| * sum(x^p))^(1/p)
    - Adapts smoothly between Average Pooling (p=1) and Max Pooling (p -> inf).
    - Initialized at p=3.0 to focus on localized knee pathology peaks (meniscal tears,
      ACL ruptures, cartilage defects) without diluting the signal with normal joint background.
    """
    def __init__(self, p=3.0, eps=1e-6, p_trainable=True):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p) if p_trainable else torch.tensor(p)
        self.eps = eps

    def forward(self, x):
        p = self.p.clamp(min=1.0, max=10.0)
        return x.clamp(min=self.eps).pow(p).mean(dim=[-2, -1]).pow(1.0 / p)



class ConvNeXtFPN(nn.Module):
    """
    🏗️ Feature Pyramid Network (FPN) Top-Down 2D Spatial Fusion (Baseline Version):
    Fuses high-level semantic context from Stage 4 (7x7) with fine-grained anatomical
    spatial detail from Stage 3 (14x14) in true 2D feature map space BEFORE pooling.

    Architecture:
    1. Lateral Conv 4: 1x1 Conv (768 -> 768) + GroupNorm + GELU
    2. Lateral Conv 3: 1x1 Conv (384 -> 768) + GroupNorm + GELU
    3. Top-Down Pathway: P4_up = Upsample2x(P4) (7x7 -> 14x14)
    4. 2D Spatial Element-wise Sum: P3 = Lateral3(Stage 3) + P4_up
    5. Anti-aliasing Smooth Conv: 3x3 Depthwise Separable Conv on P3
    6. Multi-Scale GeM Pooling: Z4 = GeMPool(P4), Z3 = GeMPool(P3)
    7. Active Residual Blending: Z_fused = Z4 + gamma * Z3 (Scalar gamma init=0.05)
    """
    def __init__(self, in_channels_s3=384, in_channels_s4=768, out_dim=768,
                 gem_p=3.0, gem_p_trainable=True, gamma_init=0.05):
        super().__init__()
        self.out_dim = out_dim

        self.lateral_s4 = nn.Sequential(
            nn.Conv2d(in_channels_s4, out_dim, kernel_size=1, bias=False),
            nn.GroupNorm(num_groups=32, num_channels=out_dim),
            nn.GELU()
        )
        self.lateral_s3 = nn.Sequential(
            nn.Conv2d(in_channels_s3, out_dim, kernel_size=1, bias=False),
            nn.GroupNorm(num_groups=32, num_channels=out_dim),
            nn.GELU()
        )

        self.smooth_s3 = nn.Sequential(
            nn.Conv2d(out_dim, out_dim, kernel_size=3, padding=1, groups=out_dim, bias=False),
            nn.Conv2d(out_dim, out_dim, kernel_size=1, bias=False),
            nn.GroupNorm(num_groups=32, num_channels=out_dim),
            nn.GELU()
        )

        self.pool_p4 = GeMPool2d(p=gem_p, eps=1e-6, p_trainable=gem_p_trainable)
        self.pool_p3 = GeMPool2d(p=gem_p, eps=1e-6, p_trainable=gem_p_trainable)
        self.gamma = nn.Parameter(torch.full((1,), float(gamma_init)))

    def forward(self, stage3_map, stage4_map):
        p4 = self.lateral_s4(stage4_map)  # (N, 768, 7, 7)
        c3 = self.lateral_s3(stage3_map)  # (N, 768, 14, 14)
        p4_up = F.interpolate(p4, size=c3.shape[-2:], mode='bilinear', align_corners=False)  # (N, 768, 14, 14)
        p3 = self.smooth_s3(c3 + p4_up)  # (N, 768, 14, 14)
        z4 = self.pool_p4(p4)  # (N, 768)
        z3 = self.pool_p3(p3)  # (N, 768)
        return z4 + self.gamma * z3


def load_convnext_backbone(model_name=CFG.backbone, drop_path_rate=getattr(CFG, "drop_path_rate", 0.08)):
    print(f"✅ Loading ConvNeXt-Tiny backbone from: {model_name} (DropPath: {drop_path_rate})")
    pretrained_path = None
    if os.path.exists(model_name):
        candidates = [os.path.join(model_name, f) for f in os.listdir(model_name) if f.endswith((".pth", ".pt", ".bin", ".safetensors"))]
        if candidates:
            pretrained_path = candidates[0]
            print(f"   📦 Found local model weights: {pretrained_path}")

    try:
        if pretrained_path:
            model = timm.create_model('convnext_tiny.in12k_ft_in1k', pretrained=False, num_classes=0, drop_path_rate=drop_path_rate)
            if pretrained_path.endswith(".safetensors"):
                from safetensors.torch import load_file
                state_dict = load_file(pretrained_path)
            else:
                state_dict = torch.load(pretrained_path, map_location="cpu")
            if "model" in state_dict: state_dict = state_dict["model"]
            if "state_dict" in state_dict: state_dict = state_dict["state_dict"]
            state_dict = {k: v for k, v in state_dict.items() if not k.startswith("head.")}
            model.load_state_dict(state_dict, strict=False)
            print("   ✅ Loaded state_dict into ConvNeXt-Tiny successfully.")
        else:
            try:
                model = timm.create_model('convnext_tiny.in12k_ft_in1k', pretrained=False, num_classes=0, drop_path_rate=drop_path_rate)
                print("   ✅ Initialized offline ConvNeXt-Tiny (checkpoint will populate all weights).")
            except Exception:
                model = timm.create_model('convnext_tiny.in12k_ft_in1k', pretrained=True, num_classes=0, drop_path_rate=drop_path_rate)
                print("   ✅ Loaded pretrained timm/convnext_tiny.in12k_ft_in1k from timm hub.")
    except Exception as e:
        print(f"⚠️ Local load note ({e}). Falling back to offline model initialization...")
        model = timm.create_model('convnext_tiny.in12k_ft_in1k', pretrained=False, num_classes=0, drop_path_rate=drop_path_rate)

    class ConvNeXtWrapper(nn.Module):
        def __init__(self, m, use_gem=getattr(CFG, 'use_gem_pooling', True),
                     gem_p=getattr(CFG, 'gem_p', 3.0),
                     gem_p_trainable=getattr(CFG, 'gem_p_trainable', True),
                     use_fpn_fusion=getattr(CFG, 'use_fpn_fusion', True),
                     use_multi_stage_fusion=getattr(CFG, 'use_multi_stage_fusion', True),
                     stage3_feat_dim=getattr(CFG, 'stage3_feat_dim', 384),
                     fpn_gamma_init=getattr(CFG, 'fpn_gamma_init', 0.05),
                     stage3_gamma_init=getattr(CFG, 'stage3_gamma_init', 0.0)):
            super().__init__()
            self.model = m
            self.embed_dim = 768
            self.use_gem = use_gem
            self.use_fpn_fusion = use_fpn_fusion
            self.use_multi_stage_fusion = use_multi_stage_fusion

            if self.use_gem:
                self.pool = GeMPool2d(p=gem_p, eps=1e-6, p_trainable=gem_p_trainable)
            else:
                self.pool = None

            # 🌟 2D Top-Down Feature Pyramid Network (FPN) Module
            if self.use_fpn_fusion:
                self.fpn = ConvNeXtFPN(
                    in_channels_s3=stage3_feat_dim, in_channels_s4=768, out_dim=768,
                    gem_p=gem_p, gem_p_trainable=gem_p_trainable, gamma_init=fpn_gamma_init
                )
            else:
                self.fpn = None
                if self.use_multi_stage_fusion:
                    self.pool_stage3 = GeMPool2d(p=gem_p, eps=1e-6, p_trainable=gem_p_trainable) if self.use_gem else None
                    self.stage3_proj = nn.Sequential(
                        nn.LayerNorm(stage3_feat_dim),
                        nn.Linear(stage3_feat_dim, 768),
                        nn.GELU()
                    )
                    self.stage3_gamma = nn.Parameter(torch.full((1,), float(stage3_gamma_init)))
                else:
                    self.pool_stage3 = None
                    self.stage3_proj = None
                    self.stage3_gamma = None

        def forward(self, x):
            if (not self.use_fpn_fusion and not self.use_multi_stage_fusion) or not hasattr(self.model, 'stages') or len(self.model.stages) < 4:
                feat_map = self.model.forward_features(x)
                if self.use_gem and self.pool is not None:
                    return self.pool(feat_map)
                else:
                    return feat_map.mean(dim=[-2, -1])

            # Sequential forward through early stages
            h = self.model.stem(x)
            h = self.model.stages[0](h)
            h = self.model.stages[1](h)
            stage3_map = self.model.stages[2](h)  # (N, 384, 14, 14)
            stage4_map = self.model.stages[3](stage3_map)  # (N, 768, 7, 7)
            if hasattr(self.model, 'norm_pre') and self.model.norm_pre is not None:
                stage4_map = self.model.norm_pre(stage4_map)

            # 🌟 2D Top-Down Spatial Fusion Pathway
            if self.use_fpn_fusion and self.fpn is not None:
                return self.fpn(stage3_map, stage4_map)

            # 1D Vector Fallback Pathway
            feat_s4 = self.pool(stage4_map) if (self.use_gem and self.pool is not None) else stage4_map.mean(dim=[-2, -1])
            feat_s3 = self.pool_stage3(stage3_map) if (self.use_gem and hasattr(self, 'pool_stage3') and self.pool_stage3 is not None) else stage3_map.mean(dim=[-2, -1])
            feat_s3_proj = self.stage3_proj(feat_s3)
            return feat_s4 + self.stage3_gamma * feat_s3_proj

    return ConvNeXtWrapper(model)

class CrossPlaneFusionBlock(nn.Module):
    def __init__(self, dim=768, num_heads=CFG.cross_plane_heads, dropout=CFG.cross_plane_dropout):
        super().__init__()
        self.pre_norm = nn.LayerNorm(dim)
        self.cross_attn = nn.MultiheadAttention(embed_dim=dim, num_heads=num_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.Linear(dim, dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * 2, dim)
        )
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, feats):
        h = self.pre_norm(feats)
        attn_out, _ = self.cross_attn(h, h, h)
        x = feats + self.gamma * attn_out
        x = self.norm2(x + self.ffn(self.norm1(x)))
        return x



class IntraPlaneVolumetricDepthBlock(nn.Module):
    """
     SOTA 3D Intra-Plane Volumetric Depth Interaction Block:
    Models true continuous through-plane anatomical connectivity (ACL path, meniscus integrity,
    joint space uniformity) along the slice depth dimension within each isolated imaging plane.
    
    Properties:
    1. Zero Cross-Plane Leakage: Reshapes (B, 3 * S_p, C) -> (B * 3, S_p, C) so planes never cross.
    2. Depthwise 1D Convolution (Kernel=3, Padding=1): Allows slice s to interact strictly with s-1 and s+1.
    3. Channel Gating & LayerNorm: Normalizes feature distribution across depth slices.
    4. Zero-Initialized Residual Gate (gamma = 0.0): Starts at 100% identity to guarantee zero degradation.
    """
    def __init__(self, dim=768, n_planes=3, dropout=0.10):
        super().__init__()
        self.n_planes = n_planes
        self.norm = nn.LayerNorm(dim)
        # Depthwise Conv: kernel_size=3 over slices
        self.depth_conv = nn.Conv1d(dim, dim, kernel_size=3, padding=1, groups=dim, bias=False)
        self.pointwise = nn.Sequential(
            nn.Conv1d(dim, dim, kernel_size=1, bias=False),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, feats):
        # feats: (B, S, C) where S = n_planes * S_p
        B, S, C = feats.shape
        S_p = S // self.n_planes
        
        # Reshape strictly within each plane: (B * 3, S_p, C)
        h = feats.view(B * self.n_planes, S_p, C)
        h_norm = self.norm(h).transpose(1, 2) # (B * 3, C, S_p)
        
        # Depthwise 3D through-plane interaction
        out = self.depth_conv(h_norm)
        out = self.pointwise(out) # (B * 3, C, S_p)
        out = out.transpose(1, 2).contiguous().view(B, S, C)
        
        return feats + self.gamma * out

class BilateralFindingRelationHead(nn.Module):
    """
    🦴 Bilateral & Cross-Finding Relational Head:
    1. Bilateral Compartmental Coupling:
       Explicitly couples anatomically symmetric compartments:
       - Pair A: Medial Meniscus (idx 2) <-> Lateral Meniscus (idx 3)
       - Pair B: Medial OA (idx 4) <-> Lateral OA (idx 5)
       Computes bilateral symmetry contrast and cross-compartmental modulation.
    2. Global Multi-Head Finding Self-Attention:
       Full (12 x 12) graph interaction across all knee findings with clinical prior bias:
       - Unhappy Triad (ACL <-> Medial Meniscus <-> Bone Contusion)
       - Degenerative Complex (Medial OA <-> Baker's Cyst <-> Effusion)
    3. Zero-Initialized Residual Gate (gamma = 0.0):
       Guarantees 100% baseline identity match at Step 0, smoothly learning optimal relation infusion.
    """
    def __init__(self, dim=768, num_classes=12, num_heads=4, dropout=0.10, gamma_init=0.0):
        super().__init__()
        self.dim = dim
        self.num_classes = num_classes

        # ── 1. Bilateral Compartmental Cross-Gating ────────────────────────
        # Medial <-> Lateral pairs: (2, 3) for Meniscus, (4, 5) for OA
        self.bilateral_pairs = [(2, 3), (4, 5)]
        self.bilateral_gate = nn.Sequential(
            nn.Linear(dim * 2, dim // 2),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(dim // 2, dim * 2),
            nn.Sigmoid()
        )

        # ── 2. Global Cross-Finding Self-Attention ─────────────────────────
        self.norm = nn.LayerNorm(dim)
        self.cross_finding_attn = nn.MultiheadAttention(
            embed_dim=dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        # Learnable Clinical Relation Prior Matrix (12, 12)
        prior_matrix = torch.zeros(num_classes, num_classes)
        # Bilateral / Symmetrical pairs
        prior_matrix[2, 3] = prior_matrix[3, 2] = 0.5   # Medial <-> Lateral Meniscus
        prior_matrix[4, 5] = prior_matrix[5, 4] = 0.5   # Medial <-> Lateral OA
        # Medial compartment complex
        prior_matrix[1, 2] = prior_matrix[2, 1] = 0.5   # MCL <-> Medial Meniscus
        prior_matrix[2, 4] = prior_matrix[4, 2] = 0.5   # Medial Meniscus <-> Medial OA
        # Degenerative / Fluid complex
        prior_matrix[4, 9] = prior_matrix[9, 4] = 0.5   # Medial OA <-> Baker's Cyst
        prior_matrix[7, 8] = prior_matrix[8, 7] = 0.5   # Effusion <-> Synovitis
        # Trauma complex (Unhappy Triad)
        prior_matrix[0, 2] = prior_matrix[2, 0] = 0.5   # ACL <-> Medial Meniscus
        prior_matrix[0, 10] = prior_matrix[10, 0] = 0.5 # ACL <-> Contusion
        prior_matrix[10, 11] = prior_matrix[11, 10] = 0.5 # Contusion <-> Fracture

        self.register_buffer("relation_prior", prior_matrix)
        self.prior_scale = nn.Parameter(torch.tensor(0.5, dtype=torch.float32))

        # Feed-forward refinement
        self.norm2 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.Linear(dim, dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim)
        )

        # ── 3. Zero-Initialized Residual Gate ──────────────────────────────
        self.gamma = nn.Parameter(torch.full((1,), float(gamma_init)))

    def forward(self, z):
        """
        Input:  z of shape (B, 12, dim) - 12 finding representation tokens
        Output: z_refined of shape (B, 12, dim)
        """
        B, N, D = z.shape

        # 1. Bilateral Compartmental Coupling
        z_bi = z.clone()
        for idx_m, idx_l in self.bilateral_pairs:
            f_m = z[:, idx_m, :]
            f_l = z[:, idx_l, :]
            f_cat = torch.cat([f_m, f_l], dim=-1)
            gates = self.bilateral_gate(f_cat)
            gate_m, gate_l = gates.chunk(2, dim=-1)
            z_bi[:, idx_m, :] = f_m + gate_m * f_l
            z_bi[:, idx_l, :] = f_l + gate_l * f_m

        # 2. Multi-Head Cross-Finding Self-Attention
        h = self.norm(z_bi)
        attn_bias = self.prior_scale * self.relation_prior
        attn_out, _ = self.cross_finding_attn(h, h, h, attn_mask=attn_bias)

        # 3. Residual FFN
        h2 = self.norm2(z_bi + attn_out)
        out = h2 + self.ffn(h2)

        # 4. Zero-Init Residual Blending
        return z + self.gamma * (out - z)

class FindingSpecificAttentionHead(nn.Module):
    def __init__(self, dim=768, n_out=12, hidden=256, dropout=CFG.dropout,
                 use_prior=CFG.use_slot_prior, prior_strength=CFG.slot_prior_strength,
                 prior_sigma=CFG.slot_prior_sigma,
                 use_cosine=getattr(CFG, "use_cosine_classifier", True),
                 use_dual_pooling=getattr(CFG, "use_dual_pooling", True),
                 focal_temperature=getattr(CFG, "focal_temperature", 0.25),
                 use_plane_decoupled_pooling=getattr(CFG, "use_plane_decoupled_pooling", True),
                 use_finding_relation=getattr(CFG, "use_finding_relation", False),
                 use_dynamic_contrast_gating=getattr(CFG, "use_dynamic_contrast_gating", False)):
        super().__init__()
        self.dim = dim
        self.n_out = n_out
        self.use_prior = use_prior
        self.use_cosine = use_cosine
        self.use_dual_pooling = use_dual_pooling
        self.focal_temperature = focal_temperature
        self.broad_temperature = getattr(CFG, 'broad_temperature', getattr(CFG, 'focal_temperature', 2.0))
        self.use_plane_decoupled_pooling = use_plane_decoupled_pooling
        self.use_finding_relation = use_finding_relation
        if self.use_finding_relation:
            self.relation_head = BilateralFindingRelationHead(dim=dim, num_classes=n_out, num_heads=getattr(CFG, "relation_num_heads", 4), dropout=getattr(CFG, "relation_dropout", 0.10), gamma_init=getattr(CFG, "relation_gamma_init", 0.0))
        else:
            self.relation_head = None
        self.use_dynamic_contrast_gating = use_dynamic_contrast_gating

        # 🌟 Contrast-Aware Dynamic Inter-Plane Gating (4 contrast types -> 12 pathology gate shifts)
        if use_dynamic_contrast_gating:
            self.contrast_plane_gate = nn.Linear(getattr(CFG, 'n_contrast_types', 4), n_out, bias=False)
            nn.init.zeros_(self.contrast_plane_gate.weight)

        # 🌟 Inter-Plane Gating Residual Parameter (zero-initialized)
        self.plane_gate = nn.Parameter(torch.zeros(n_out, 3, dtype=torch.float32))

        self.norm = nn.LayerNorm(dim)
        self.att = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(hidden, n_out)
        )
        self.clsW = nn.Parameter(torch.zeros(n_out, dim))
        self.clsb = nn.Parameter(torch.zeros(n_out))
        nn.init.trunc_normal_(self.clsW, std=0.02)

        if use_cosine:
            # Learnable Temperature Scale initialized to log(16.0)
            self.logit_scale = nn.Parameter(torch.tensor(2.7725887, dtype=torch.float32)) # ln(16.0)

        # 🌟 Dual-Pooling: Pathology-specific zero-initialized residual focal gate
        if use_dual_pooling:
            self.focal_gate = nn.Parameter(torch.zeros(n_out, 1, dtype=torch.float32))
        else:
            self.focal_gate = None

        if use_prior:
            s_per_plane = getattr(CFG, "num_slices_per_plane", 7)
            self.learnable_slot_prior = WarmStartResidualGaussianSlotPrior(n_classes=n_out, n_slices_per_plane=s_per_plane)
            init_val = float(np.log(np.exp(prior_strength) - 1.0))
            self.prior_scale = nn.Parameter(torch.ones(1, 1, n_out, dtype=torch.float32) * init_val)
        else:
            self.learnable_slot_prior = None
            self.prior_scale = None

    def forward(self, feats, plane_mask=None, c_plane_dist=None):
        h = self.norm(feats)                            # (B, 21, 768)
        a_logits = self.att(h)                          # (B, 21, 12)
        if self.use_prior and self.learnable_slot_prior is not None:
            effective_scale = F.softplus(self.prior_scale)
            slot_prior = self.learnable_slot_prior()
            a_logits = a_logits + effective_scale * slot_prior.unsqueeze(0)

        if plane_mask is not None:
            if plane_mask.dim() == 2:
                plane_mask = plane_mask.unsqueeze(-1)
            a_logits = a_logits.masked_fill(plane_mask, -1e4)

        if not getattr(self, "use_plane_decoupled_pooling", False):
            # Monolithic 21-element Baseline
            a_ctx = torch.softmax(a_logits, dim=1)          # (B, 21, 12)
            pooled = torch.einsum("bsn, bsf -> bnf", a_ctx, h) # (B, 12, 768)

            if self.use_dual_pooling and self.focal_gate is not None:
                # 🌟 Dual-Scale Context: soft smoothed attention across all slices (tau=2.0)
                tau_broad = max(getattr(self, "broad_temperature", getattr(self, "focal_temperature", 2.0)), 1e-3)
                a_broad = torch.softmax(a_logits / tau_broad, dim=1) # (B, 21, 12)
                pooled_broad = torch.einsum("bsn, bsf -> bnf", a_broad, h) # (B, 12, 768)
                #  Linear Focal Gating (Zero-Init: Starts at 100% Focal, smoothly learns Broad context without 50% prior bias)
                pooled = pooled + self.focal_gate * (pooled_broad - pooled)
        else:
            # 🌟 Plane-Decoupled Hierarchical Dual-Pooling (Intra-Plane + Inter-Plane Gating)
            B, S, F_dim = h.shape
            P = 3
            S_p = S // P  # 7 slices per plane
            h_3d = h.view(B, P, S_p, F_dim)
            a_logits_3d = a_logits.view(B, P, S_p, self.n_out)

            # Level 1: Intra-Plane Attention Pooling (dim=2: softmax over 7 slices within each plane)
            a_ctx_intra = torch.softmax(a_logits_3d, dim=2) # (B, P=3, S_p=7, N=12)
            z_ctx = torch.einsum("bpsn, bpsf -> bpnf", a_ctx_intra, h_3d) # (B, P=3, N=12, F=768)

            if self.use_dual_pooling and self.focal_gate is not None:
                # 🌟 Dual-Scale Context: soft smoothed attention across all slices (tau=2.0)
                tau_broad = max(getattr(self, "broad_temperature", getattr(self, "focal_temperature", 2.0)), 1e-3)
                a_broad_intra = torch.softmax(a_logits_3d / tau_broad, dim=2) # (B, P=3, S_p=7, N=12)
                z_broad = torch.einsum("bpsn, bpsf -> bpnf", a_broad_intra, h_3d) # (B, P=3, N=12, F=768)
                #  Linear Focal Gating (Zero-Init: Starts at 100% Focal, smoothly learns Broad context without 50% prior bias)
                z_intra = z_ctx + self.focal_gate.view(1, 1, self.n_out, 1) * (z_broad - z_ctx)
            else:
                z_intra = z_ctx

            # Level 2: Inter-Plane Gating
            if self.use_prior and self.learnable_slot_prior is not None:
                _, _, pi = self.learnable_slot_prior.get_geometry() # (12, 3)
                w_logits = pi + self.plane_gate                     # (12, 3)
            else:
                w_logits = self.plane_gate                          # (12, 3)

            w_logits = w_logits.unsqueeze(0).expand(B, -1, -1)     # (B, 12, 3)

            # 🌟 Dynamic Contrast-Aware Inter-Plane Shift
            if getattr(self, 'use_dynamic_contrast_gating', False) and c_plane_dist is not None:
                # c_plane_dist: (B, 3, 4) -> map each plane's contrast to 12 disease affinities -> (B, 3, 12) -> (B, 12, 3)
                delta_w = self.contrast_plane_gate(c_plane_dist).permute(0, 2, 1)
                gamma = getattr(CFG, 'dynamic_plane_gamma', 1.0)
                w_logits = w_logits + gamma * delta_w

            # Mask out dropped plane if Whole-Plane Dropout occurred
            if plane_mask is not None:
                dropped_planes = plane_mask.view(B, P, S_p).all(dim=-1) # (B, 3)
                if dropped_planes.any():
                    w_logits = w_logits.masked_fill(dropped_planes.unsqueeze(1), -1e4)

            w_plane = torch.softmax(w_logits, dim=-1) # (B, 12, 3) - sum over 3 planes = 1.0
            pooled = torch.einsum("bpnf, bnp -> bnf", z_intra, w_plane) # (B, 12, 768)

        # 🌟 Bilateral Finding Relation Head
        if getattr(self, "use_finding_relation", False) and self.relation_head is not None:
            pooled = self.relation_head(pooled)

        if self.use_cosine:
            pooled_norm = F.normalize(pooled, p=2, dim=-1, eps=1e-6)   # (B, 12, 768)
            clsW_norm   = F.normalize(self.clsW,   p=2, dim=-1, eps=1e-6)   # (12, 768)
            scale = torch.exp(torch.clamp(self.logit_scale, max=4.158883))               # max scale = 64.0
            cos_sim = (pooled_norm * clsW_norm.unsqueeze(0)).sum(-1)    # (B, 12)
            logits  = scale * cos_sim + self.clsb
        else:
            logits = (pooled * self.clsW).sum(-1) + self.clsb          # (B, 12)

        return logits


class ConvNeXt25DModel(nn.Module):
    def __init__(self, model_name=CFG.backbone, num_classes=12, unfreeze_last=CFG.unfreeze_last,
                 head_type=CFG.head_type, use_plane_embedding=CFG.use_plane_embedding,
                 use_contrast_embedding=CFG.use_contrast_embedding,
                 use_contrast_aux=CFG.use_contrast_aux,
                 n_contrast=CFG.n_contrast_types,
                 use_cross_plane=getattr(CFG, "use_cross_plane", False),
                 use_plane_dropout=CFG.use_plane_dropout,
                 plane_dropout_prob=CFG.plane_dropout_prob,
                 use_metadata_injection=getattr(CFG, "use_metadata_injection", True),
                 meta_dim=getattr(CFG, "meta_dim", 13)):
        super().__init__()
        self.backbone = load_convnext_backbone(model_name, drop_path_rate=getattr(CFG, "drop_path_rate", 0.08))
        feat_dim = self.backbone.embed_dim      # 768 dims
        self.use_plane_embedding = use_plane_embedding
        self.use_contrast_embedding = use_contrast_embedding
        self.use_contrast_aux = use_contrast_aux
        self.n_contrast = n_contrast
        self.use_cross_plane = use_cross_plane
        self.use_intra_plane_depth = getattr(CFG, "use_intra_plane_depth", False)
        if self.use_intra_plane_depth:
            self.depth_block = IntraPlaneVolumetricDepthBlock(dim=feat_dim, n_planes=3, dropout=getattr(CFG, "depth_interact_dropout", 0.10))
        else:
            self.depth_block = None
        self.use_plane_dropout = use_plane_dropout
        self.plane_dropout_prob = plane_dropout_prob
        self.use_metadata_injection = use_metadata_injection

        if use_plane_embedding:
            self.plane_emb = nn.Parameter(torch.randn(3, feat_dim) * 0.02)
            s_p = getattr(CFG, 'num_slices_per_plane', 7)
            plane_idx = torch.tensor([0]*s_p + [1]*s_p + [2]*s_p, dtype=torch.long)
            self.register_buffer("plane_idx", plane_idx)

        if use_contrast_embedding:
            self.contrast_emb = nn.Embedding(n_contrast, feat_dim)
            nn.init.trunc_normal_(self.contrast_emb.weight, std=0.02)

        # 🌟 Auxiliary Contrast Prediction Head
        if use_contrast_aux:
            self.contrast_head = nn.Sequential(
                nn.LayerNorm(feat_dim),
                nn.Linear(feat_dim, 64),
                nn.GELU(),
                nn.Dropout(0.10),
                nn.Linear(64, n_contrast)
            )

        # 🌟 13-Dim Tabular Patient & Scanner Metadata Injection Modules
        if self.use_metadata_injection:
            self.use_meta_film = getattr(CFG, "use_meta_film", False)
            if self.use_meta_film:
                self.meta_encoder = nn.Sequential(
                    nn.Linear(meta_dim, 128),
                    nn.LayerNorm(128),
                    nn.SiLU(),
                    nn.Dropout(getattr(CFG, "meta_mlp_dropout", 0.20)),
                    nn.Linear(128, feat_dim)
                )
                self.meta_gamma = nn.Parameter(torch.zeros(1))  # Zero-init gating for smooth warm-up
            else:
                self.meta_encoder = None
                self.meta_gamma = None

            self.meta_finding_bias = nn.Sequential(
                nn.Linear(meta_dim, 64),
                nn.LayerNorm(64),
                nn.SiLU(),
                nn.Dropout(getattr(CFG, "meta_mlp_dropout", 0.20)),
                nn.Linear(64, num_classes)
            )
        else:
            self.use_meta_film = False
            self.meta_encoder = None
            self.meta_gamma = None
            self.meta_finding_bias = None

        if use_cross_plane:
            self.cross_plane = CrossPlaneFusionBlock(dim=feat_dim, num_heads=CFG.cross_plane_heads, dropout=CFG.cross_plane_dropout)
        else:
            self.cross_plane = None

        self.head = FindingSpecificAttentionHead(dim=feat_dim, n_out=num_classes)

        # Freeze early stages, unfreeze top stages
        for param in self.backbone.parameters():
            param.requires_grad = False
        bb_module = self.backbone.model if hasattr(self.backbone, "model") else self.backbone

        # timm ConvNeXt structure: stages[0], stages[1], stages[2], stages[3]
        if hasattr(bb_module, "stages"):
            for stage in bb_module.stages[-unfreeze_last:]:
                for param in stage.parameters():
                    param.requires_grad = True
            if hasattr(bb_module, "norm_pre"):
                for param in bb_module.norm_pre.parameters(): param.requires_grad = True
            if hasattr(bb_module, "head"):
                for param in bb_module.head.parameters(): param.requires_grad = True
        else:
            for param in list(bb_module.parameters())[-30:]:
                param.requires_grad = True

        # 🌟 Explicitly Unfreeze GeM Pooling & Multi-Stage Feature Fusion in Backbone Wrapper!
        if hasattr(self.backbone, "pool") and self.backbone.pool is not None:
            for param in self.backbone.pool.parameters():
                param.requires_grad = True
        if hasattr(self.backbone, "pool_stage3") and self.backbone.pool_stage3 is not None:
            for param in self.backbone.pool_stage3.parameters():
                param.requires_grad = True
        if hasattr(self.backbone, "stage3_proj") and self.backbone.stage3_proj is not None:
            for param in self.backbone.stage3_proj.parameters():
                param.requires_grad = True
        if hasattr(self.backbone, "stage3_gamma") and self.backbone.stage3_gamma is not None:
            self.backbone.stage3_gamma.requires_grad = True
        if hasattr(self.backbone, 'fpn') and self.backbone.fpn is not None:
            for param in self.backbone.fpn.parameters():
                param.requires_grad = True

    def forward(self, x, contrast_flags=None, meta_vec=None):
        B, S, C, H, W = x.shape
        x_flat = x.view(B * S, C, H, W)
        if getattr(CFG, "use_channels_last", True):
            x_flat = x_flat.contiguous(memory_format=torch.channels_last)
        chunk_size = getattr(CFG, "backbone_chunk_size", 672)
        if x_flat.size(0) > chunk_size and self.training:
            raw_feats = torch.cat([self.backbone(c) for c in x_flat.split(chunk_size)], dim=0)
        else:
            raw_feats = self.backbone(x_flat)
        raw_feats = raw_feats.view(B, S, -1)                # (B, S, 768)

        aux_logits = None
        if self.use_contrast_aux:
            aux_logits = self.contrast_head(raw_feats)       # (B, 15, 4)

        feats = raw_feats

        if self.use_plane_embedding:
            feats = feats + self.plane_emb[self.plane_idx].unsqueeze(0)

        if self.use_contrast_embedding and contrast_flags is not None:
            c_clamped = contrast_flags.clamp(0, self.n_contrast - 1)
            feats = feats + self.contrast_emb(c_clamped)

        # 🌟 Feature-Level Metadata FiLM Injection with Modality Dropout
        meta_drop_mask = None
        if self.use_metadata_injection and meta_vec is not None:
            if self.training and getattr(CFG, "meta_dropout_prob", 0.0) > 0:
                # Modality Dropout: 25% of train samples get 0 metadata contribution, forcing 100% reliance on visual features
                meta_drop_mask = (torch.rand(meta_vec.size(0), 1, device=meta_vec.device) > CFG.meta_dropout_prob).float()

            if getattr(self, "use_meta_film", False) and self.meta_encoder is not None:
                meta_feat = self.meta_encoder(meta_vec).unsqueeze(1) # (B, 1, 768)
                if meta_drop_mask is not None:
                    meta_feat = meta_feat * meta_drop_mask.unsqueeze(1) # (B, 1, 1)
                feats = feats + self.meta_gamma * meta_feat

        # 🛡️ EXP 3.2: Pre-Fusion Whole-Plane Dropout (Zero Information Leakage)
        plane_mask = None
        if self.training and self.use_plane_dropout and self.plane_dropout_prob > 0:
            mask_tensor = torch.zeros((B, CFG.num_slices, 1), dtype=torch.bool, device=feats.device)
            s_per_plane = getattr(CFG, "num_slices_per_plane", 7)
            for b_idx in range(B):
                if random.random() < self.plane_dropout_prob:
                    drop_plane = random.randint(0, 2)
                    start_s = drop_plane * s_per_plane
                    end_s   = start_s + s_per_plane
                    mask_tensor[b_idx, start_s:end_s, :] = True
            if mask_tensor.any():
                plane_mask = mask_tensor
                # Zero out dropped plane features BEFORE CrossPlaneFusion to prevent leakage across attention!
                feats = feats.masked_fill(plane_mask, 0.0)

        if self.use_cross_plane and self.cross_plane is not None:
            feats = self.cross_plane(feats)

        # 🌟 Compute Plane-Specific Contrast Distribution (B, 3, 4) for Dynamic Inter-Plane Gating
        c_plane_dist = None
        if self.use_contrast_aux and aux_logits is not None:
            s_p = getattr(CFG, 'num_slices_per_plane', S // 3)
            c_plane_dist = torch.softmax(aux_logits.view(B, 3, s_p, self.n_contrast), dim=-1).mean(dim=2) # (B, 3, 4)

        #  Intra-Plane 3D Volumetric Depth Interaction (Before Hierarchical Attention Pooling)
        if self.use_intra_plane_depth and self.depth_block is not None:
            feats = self.depth_block(feats)

        main_logits = self.head(feats, plane_mask=plane_mask, c_plane_dist=c_plane_dist)  # (B, 12)

        # 🌟 Logit-Level Clinical Prior Bias Injection (with Modality Dropout + Reduced Weight)
        if self.use_metadata_injection and meta_vec is not None:
            bias = self.meta_finding_bias(meta_vec)
            if meta_drop_mask is not None:
                bias = bias * meta_drop_mask
            bias_w = getattr(CFG, "meta_logit_bias_weight", 0.03)
            main_logits = main_logits + bias_w * bias

        if self.use_contrast_aux:
            return main_logits, aux_logits
        return main_logits

print(f"✅ ConvNeXt-Tiny Model initialized with Cosine Classifier, Metadata ({CFG.meta_dim}-dim), Aux Contrast & Contrast-Aware Dynamic Plane Gating.")




✅ ConvNeXt-Tiny Model initialized with Cosine Classifier, Metadata (13-dim), Aux Contrast & Contrast-Aware Dynamic Plane Gating.


In [6]:
import os, gc
import numpy as np
import torch

# =========================================================================
# 6. Multi-Fold Model Preloading (Concurrent GPU VRAM Caching)
# =========================================================================

models_ensemble = []
loaded_checkpoint_paths = []

for fold in range(5):
    # Candidate filenames ordered by priority
    possible_names = [
        f"best_convnext_tiny_kaggle_fold{fold}.pth",
        f"best_convnext_tiny_colab_fold{fold}.pth",
        f"swa_convnext_tiny_kaggle_fold{fold}.pth",
        f"swa_convnext_tiny_colab_fold{fold}.pth",
        f"convnext_tiny_kaggle_fold{fold}.pth",
        f"convnext_tiny_fold{fold}.pth",
        f"convnext_fold{fold}.pth",
        f"fold{fold}.pth",
        f"fold_{fold}.pth"
    ]
    ckpt_path = None
    
    # 1. Search in CFG.ckpt_candidates
    candidate_search_dirs = getattr(CFG, "ckpt_candidates", [getattr(CFG, "ckpt_dir", "./checkpoints")])
    for s_dir in candidate_search_dirs:
        if os.path.exists(s_dir):
            for p_name in possible_names:
                cand = os.path.join(s_dir, p_name)
                if os.path.exists(cand):
                    ckpt_path = cand
                    break
            if ckpt_path is None:
                for root, _, files in os.walk(s_dir):
                    for f in files:
                        if f.endswith(".pth") and (f"fold{fold}" in f.lower() or f"fold_{fold}" in f.lower()):
                            ckpt_path = os.path.join(root, f)
                            break
                    if ckpt_path: break
        if ckpt_path: break

    # 2. Search in local checkpoints directory
    if ckpt_path is None:
        for local_dir in ["./checkpoints", "."]:
            for p_name in possible_names:
                cand = os.path.join(local_dir, p_name)
                if os.path.exists(cand):
                    ckpt_path = cand
                    break
            if ckpt_path: break

    # 3. ⚡ Robust Fallback: Scan all /kaggle/input/ datasets
    if ckpt_path is None and os.path.exists("/kaggle/input"):
        for root, _, files in os.walk("/kaggle/input"):
            if any(skip_dir in root for skip_dir in ["test_series", "test_images", "train_series", "train_images"]):
                continue
            for f in files:
                if f.endswith(".pth") and (f"fold{fold}" in f.lower() or f"fold_{fold}" in f.lower()):
                    ckpt_path = os.path.join(root, f)
                    break
            if ckpt_path: break

    if ckpt_path is None:
        print(f"⚠️ Warning: Checkpoint for Fold {fold} not found in any search path!")
        continue

    print(f"📦 Preloading Fold {fold} Checkpoint from: {ckpt_path}")
    checkpoint = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    
    # Handle state_dict unpacking & Auto-sync Config from Checkpoint
    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        state_dict = checkpoint['model_state_dict']
        val_auc = checkpoint.get('best_auc', checkpoint.get('val_macro_auc', 'N/A'))
        is_swa = checkpoint.get('swa', False)
        auc_str = f"{val_auc:.4f}" if isinstance(val_auc, float) else str(val_auc)
        print(f"   ✅ Loaded Checkpoint (Val Macro-AUC: {auc_str} | SWA: {is_swa})")
        
        # 🌟 Auto-synchronize runtime configuration from trained checkpoint
        if 'config' in checkpoint and isinstance(checkpoint['config'], dict):
            ckpt_cfg = checkpoint['config']
            if 'meta_logit_bias_weight' in ckpt_cfg:
                CFG.meta_logit_bias_weight = float(ckpt_cfg['meta_logit_bias_weight'])
            if 'use_slot_prior' in ckpt_cfg:
                CFG.use_slot_prior = bool(ckpt_cfg['use_slot_prior'])
            if 'use_cosine_classifier' in ckpt_cfg:
                CFG.use_cosine_classifier = bool(ckpt_cfg['use_cosine_classifier'])
            if 'head_type' in ckpt_cfg:
                CFG.head_type = str(ckpt_cfg['head_type'])
            if 'use_cross_plane' in ckpt_cfg:
                CFG.use_cross_plane = bool(ckpt_cfg['use_cross_plane'])
            if 'use_intra_plane_depth' in ckpt_cfg:
                CFG.use_intra_plane_depth = bool(ckpt_cfg['use_intra_plane_depth'])
            if 'depth_interact_dropout' in ckpt_cfg:
                CFG.depth_interact_dropout = float(ckpt_cfg['depth_interact_dropout'])
            if 'use_meta_film' in ckpt_cfg:
                CFG.use_meta_film = bool(ckpt_cfg['use_meta_film'])
            if 'use_contrast_aux' in ckpt_cfg:
                CFG.use_contrast_aux = bool(ckpt_cfg['use_contrast_aux'])
            if 'meta_dim' in ckpt_cfg:
                CFG.meta_dim = int(ckpt_cfg['meta_dim'])
            if 'use_plane_embedding' in ckpt_cfg:
                CFG.use_plane_embedding = bool(ckpt_cfg['use_plane_embedding'])
            if 'use_contrast_embedding' in ckpt_cfg:
                CFG.use_contrast_embedding = bool(ckpt_cfg['use_contrast_embedding'])
            if 'num_slices' in ckpt_cfg:
                CFG.total_slices = int(ckpt_cfg['num_slices'])
            if 'num_slices_per_plane' in ckpt_cfg:
                CFG.num_slices_per_plane = int(ckpt_cfg['num_slices_per_plane'])
            if 'img_size' in ckpt_cfg:
                CFG.img_size = int(ckpt_cfg['img_size'])
            if 'use_multi_prototype' in ckpt_cfg:
                CFG.use_multi_prototype = bool(ckpt_cfg['use_multi_prototype'])
            if 'num_prototypes' in ckpt_cfg:
                CFG.num_prototypes = int(ckpt_cfg['num_prototypes'])
            if 'use_dual_pooling' in ckpt_cfg:
                CFG.use_dual_pooling = bool(ckpt_cfg['use_dual_pooling'])
            if 'use_plane_decoupled_pooling' in ckpt_cfg:
                CFG.use_plane_decoupled_pooling = bool(ckpt_cfg['use_plane_decoupled_pooling'])
                CFG.use_hierarchical_pooling = CFG.use_plane_decoupled_pooling
            if 'focal_temperature' in ckpt_cfg:
                CFG.focal_temperature = float(ckpt_cfg['focal_temperature'])
            if 'broad_temperature' in ckpt_cfg:
                CFG.broad_temperature = float(ckpt_cfg['broad_temperature'])
            if 'use_dynamic_contrast_gating' in ckpt_cfg:
                CFG.use_dynamic_contrast_gating = bool(ckpt_cfg['use_dynamic_contrast_gating'])
            if 'dynamic_plane_gamma' in ckpt_cfg:
                CFG.dynamic_plane_gamma = float(ckpt_cfg['dynamic_plane_gamma'])
            if 'use_multi_stage_fusion' in ckpt_cfg:
                CFG.use_multi_stage_fusion = bool(ckpt_cfg['use_multi_stage_fusion'])
            if 'use_fpn_fusion' in ckpt_cfg:
                CFG.use_fpn_fusion = bool(ckpt_cfg['use_fpn_fusion'])
            if 'use_finding_relation' in ckpt_cfg:
                CFG.use_finding_relation = bool(ckpt_cfg['use_finding_relation'])
            if 'use_gem_pooling' in ckpt_cfg:
                CFG.use_gem_pooling = bool(ckpt_cfg['use_gem_pooling'])
            if 'use_metadata_injection' in ckpt_cfg:
                CFG.use_metadata_injection = bool(ckpt_cfg['use_metadata_injection'])
    else:
        state_dict = checkpoint

    # ⚡ Clean "module." prefix if saved with torch.nn.DataParallel
    clean_state_dict = {}
    for k, v in state_dict.items():
        name = k[7:] if k.startswith("module.") else k
        clean_state_dict[name] = v
    state_dict = clean_state_dict

    # 🔍 Auto-detect architectural components directly from state_dict keys
    has_fpn = any("fpn." in k for k in state_dict.keys())
    has_relation = any("relation_head." in k for k in state_dict.keys())
    has_dynamic_gating = any("contrast_plane_gate" in k for k in state_dict.keys())
    has_contrast_emb = any('contrast_emb' in k for k in state_dict.keys())
    has_depth_block = any("depth_block." in k for k in state_dict.keys())
    has_cross_plane = any("cross_plane." in k for k in state_dict.keys())
    has_meta_film = any("meta_encoder." in k for k in state_dict.keys())
    has_plane_emb = any("plane_emb" in k for k in state_dict.keys())
    has_meta_inj = any("meta_finding_bias" in k for k in state_dict.keys())
    has_stage3_fusion = any("stage3_gamma" in k for k in state_dict.keys())

    CFG.use_fpn_fusion = has_fpn
    CFG.use_finding_relation = has_relation
    CFG.use_dynamic_contrast_gating = has_dynamic_gating
    CFG.use_intra_plane_depth = has_depth_block
    CFG.use_cross_plane = has_cross_plane
    CFG.use_meta_film = has_meta_film
    CFG.use_plane_embedding = has_plane_emb
    CFG.use_metadata_injection = has_meta_inj
    CFG.use_multi_stage_fusion = has_stage3_fusion
    CFG.use_stage3_fusion = has_stage3_fusion
    use_contrast_emb = CFG.use_contrast_embedding or has_contrast_emb

    m_dim = CFG.meta_dim
    if 'config' in checkpoint and isinstance(checkpoint['config'], dict) and 'meta_dim' in checkpoint['config']:
        m_dim = int(checkpoint['config']['meta_dim'])

    print(f"   ⚙️ Config: FPN={CFG.use_fpn_fusion}, DepthBlock={CFG.use_intra_plane_depth}, Stage3={CFG.use_multi_stage_fusion}, PlaneEmb={CFG.use_plane_embedding}, MetaInj={CFG.use_metadata_injection}, RelationHead={CFG.use_finding_relation}, DualPool={CFG.use_dual_pooling}")

    model = ConvNeXt25DModel(
        model_name=getattr(CFG, 'model_path', getattr(CFG, 'backbone', 'convnext_tiny')),
        num_classes=len(TARGET_COLS),
        unfreeze_last=CFG.unfreeze_last,
        head_type=CFG.head_type,
        use_plane_embedding=CFG.use_plane_embedding,
        use_contrast_embedding=use_contrast_emb,
        use_contrast_aux=getattr(CFG, 'use_contrast_aux', True),
        use_cross_plane=CFG.use_cross_plane,
        use_metadata_injection=CFG.use_metadata_injection,
        meta_dim=m_dim,
    )
    if getattr(CFG, "use_channels_last", False):
        model.to(memory_format=torch.channels_last)

    try:
        model.load_state_dict(state_dict, strict=True)
        print(f"   ✅ 100% EXACT State Dict Match for Fold {fold} (0 missing, 0 unexpected keys)!")
    except Exception as e:
        print(f"   ⚠️ strict=True failed: {e}")
        missing, unexpected = model.load_state_dict(state_dict, strict=False)
        print(f"      ❌ Missing keys ({len(missing)}): {missing}")
        print(f"      ❌ Unexpected keys ({len(unexpected)}): {unexpected}")
        if len(missing) > 0 or len(unexpected) > 0:
            raise RuntimeError(f"Architecture mismatch on Fold {fold}! Please inspect model definition.")
        
    model.to(CFG.device)
    model.eval()
    models_ensemble.append(model)
    loaded_checkpoint_paths.append(ckpt_path)

if len(models_ensemble) == 0:
    raise RuntimeError(f"❌ Failed to load any fold checkpoints! Please attach your 5-fold ConvNeXt checkpoints dataset.")

print(f"\n🎉 Successfully Preloaded {len(models_ensemble)} Fold Models into GPU VRAM!")
if torch.cuda.is_available():
    vram_mb = torch.cuda.memory_allocated() / (1024 * 1024)
    print(f"📊 Total VRAM Occupied: {vram_mb:.1f} MB (< 3% of 16GB GPU)")


📦 Preloading Fold 0 Checkpoint from: /kaggle/input/datasets/mediumguy/rsna-convnext-tiny-2-stage/best_convnext_tiny_kaggle_fold0.pth
   ✅ Loaded Checkpoint (Val Macro-AUC: 0.9231 | SWA: True)
   ⚙️ Config: FPN=True, DepthBlock=False, Stage3=False, PlaneEmb=False, MetaInj=False, RelationHead=False, DualPool=True
✅ Loading ConvNeXt-Tiny backbone from: convnext_tiny.in12k_ft_in1k (DropPath: 0.08)
   ✅ Initialized offline ConvNeXt-Tiny (checkpoint will populate all weights).
   ✅ 100% EXACT State Dict Match for Fold 0 (0 missing, 0 unexpected keys)!
📦 Preloading Fold 1 Checkpoint from: /kaggle/input/datasets/mediumguy/rsna-convnext-tiny-2-stage/best_convnext_tiny_kaggle_fold1.pth
   ✅ Loaded Checkpoint (Val Macro-AUC: 0.9238 | SWA: False)
   ⚙️ Config: FPN=True, DepthBlock=False, Stage3=False, PlaneEmb=False, MetaInj=False, RelationHead=False, DualPool=True
✅ Loading ConvNeXt-Tiny backbone from: convnext_tiny.in12k_ft_in1k (DropPath: 0.08)
   ✅ Initialized offline ConvNeXt-Tiny (checkpoint

In [7]:
import os, gc
import numpy as np
import torch
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor

# =========================================================================
# 7. Zero-RAM Streaming Inference Pipeline (3-Pass Slice-Jittering TTA)
# =========================================================================

test_study_ids = sub_df["StudyInstanceUID"].tolist()
N_test = len(test_study_ids)
final_predictions = np.zeros((N_test, len(TARGET_COLS)), dtype=np.float32)
batch_size = CFG.batch_size
s_p = getattr(CFG, "num_slices_per_plane", 7)

# ⚡ GPU Vectorized Normalization Constants (0.450 / 0.225 Grayscale MRI)
GPU_MEAN = torch.tensor([0.450, 0.450, 0.450], device=CFG.device).view(1, 1, 3, 1, 1)
GPU_STD  = torch.tensor([0.225, 0.225, 0.225], device=CFG.device).view(1, 1, 3, 1, 1)

print(f"🚀 Streaming Inference over {N_test:,} Hidden Test Studies ({len(models_ensemble)} Fold Models)...\n" 
      f"   Pipeline: 21 Slices @ {CFG.img_size}x{CFG.img_size} | Slices/plane: {s_p} | Batch size: {batch_size} | Slice-Jitter TTA: {CFG.use_slice_jitter_tta} | Anatomical Flip TTA: {CFG.use_anatomical_flip_tta}")

with torch.inference_mode():
    for batch_start in tqdm(range(0, N_test, batch_size), desc="Inferring Batches"):
        batch_end = min(batch_start + batch_size, N_test)
        batch_uids = test_study_ids[batch_start:batch_end]
        B_curr = len(batch_uids)
        
        # Parallel DICOM reading across CPU threads directly from hardcoded test_series
        with ThreadPoolExecutor(max_workers=CFG.num_workers) as executor:
            results = list(executor.map(
                lambda uid: process_study(uid, study_plane_map, CFG.comp_dir, CFG.img_size, CFG.use_slice_jitter_tta),
                batch_uids
            ))
            
        batch_passes = [r[0] for r in results]
        batch_lats   = [r[1] for r in results]
        batch_cflags = [r[2] for r in results]
        batch_metas  = [r[3] for r in results]
        n_passes = len(batch_passes[0])
        
        study_preds_accumulator = np.zeros((B_curr, len(TARGET_COLS)), dtype=np.float32)
        total_weights = 0.0
        
        c_flags_tensor = torch.tensor(batch_cflags, dtype=torch.long, device=CFG.device)
        meta_tensor    = torch.tensor(np.stack(batch_metas, axis=0), dtype=torch.float32, device=CFG.device)
        
        # TTA Weights for (Standard=1.0, Forward=0.5, Backward=0.5)
        pass_weights = [1.0, 0.5, 0.5] if n_passes == 3 else [1.0]
        
        for pass_idx in range(n_passes):
            pass_weight = pass_weights[pass_idx]
            pass_vol = np.stack([p[pass_idx] for p in batch_passes], axis=0)  # (B, 21, 3, 224, 224)
            t_vol = torch.from_numpy(pass_vol).float() / 255.0
            
            # ⚡ Transfer to GPU (<1ms)
            tensor_batch = t_vol.to(CFG.device, non_blocking=True)
            
            # 🏥 1. Vectorized GPU Laterality Normalization (Right 'R' -> Standard Left 'L')
            # Strictly matches gpu_normalize_val_batch from rsna_convnext_exp.ipynb (<0.05ms)
            is_right = torch.tensor([lat == "R" for lat in batch_lats], dtype=torch.bool, device=CFG.device)
            if CFG.use_laterality_flip and is_right.any():
                # Coronal (7:14) & Axial (14:21) -> flip width (dims=[-1])
                tensor_batch[is_right, s_p:s_p*3] = torch.flip(tensor_batch[is_right, s_p:s_p*3], dims=[-1])
                # Sagittal (0:7) -> flip slice sequence (dims=1) AND channels (dims=2)
                tensor_batch[is_right, 0:s_p]     = torch.flip(tensor_batch[is_right, 0:s_p], dims=[1, 2])
                        
            # Apply GPU Grayscale Normalization
            tensor_batch = (tensor_batch - GPU_MEAN) / GPU_STD
            
            # Evaluate all preloaded fold models on standard Left-normalized view
            pass_probs = np.zeros((B_curr, len(TARGET_COLS)), dtype=np.float32)
            for model in models_ensemble:
                with torch.amp.autocast("cuda", enabled=(CFG.device.type == "cuda" and CFG.use_amp)):
                    out = model(tensor_batch, contrast_flags=c_flags_tensor, meta_vec=meta_tensor)
                    logits = out[0] if isinstance(out, tuple) else out
                    probs = torch.sigmoid(logits).cpu().numpy()
                pass_probs += probs / len(models_ensemble)
            
            # 🏥 2. Anatomical Horizontal Flip TTA (Medial <-> Lateral Swapping)
            if CFG.use_anatomical_flip_tta:
                flip_batch = tensor_batch.clone()
                # Flip Coronal & Axial width
                flip_batch[:, s_p:s_p*3] = torch.flip(flip_batch[:, s_p:s_p*3], dims=[-1])
                # Flip Sagittal slice sequence & channels
                flip_batch[:, 0:s_p]     = torch.flip(flip_batch[:, 0:s_p], dims=[1, 2])
                
                # 🔄 Invert Laterality flag (index 8: 1.0=R, 0.0=L) on mirrored knee
                meta_flipped = meta_tensor.clone()
                meta_flipped[:, 8] = 1.0 - meta_flipped[:, 8]
                
                flip_probs = np.zeros((B_curr, len(TARGET_COLS)), dtype=np.float32)
                for model in models_ensemble:
                    with torch.amp.autocast("cuda", enabled=(CFG.device.type == "cuda" and CFG.use_amp)):
                        out = model(flip_batch, contrast_flags=c_flags_tensor, meta_vec=meta_flipped)
                        logits = out[0] if isinstance(out, tuple) else out
                        probs = torch.sigmoid(logits).cpu().numpy()
                    flip_probs += probs / len(models_ensemble)
                
                # 🔄 Anatomical Swap on Mirrored Knee: 
                # Col 2 ('Medial Meniscus') <-> Col 3 ('Lateral Meniscus')
                # Col 4 ('Medial OA')       <-> Col 5 ('Lateral OA')
                flip_probs_swapped = flip_probs.copy()
                flip_probs_swapped[:, 2] = flip_probs[:, 3]
                flip_probs_swapped[:, 3] = flip_probs[:, 2]
                flip_probs_swapped[:, 4] = flip_probs[:, 5]
                flip_probs_swapped[:, 5] = flip_probs[:, 4]
                
                # Combine standard view and anatomically-swapped mirrored view (50% / 50%)
                pass_probs = 0.5 * pass_probs + 0.5 * flip_probs_swapped
                del flip_batch, meta_flipped
                
            study_preds_accumulator += pass_weight * pass_probs
            total_weights += pass_weight
            del tensor_batch, t_vol
            
        final_predictions[batch_start:batch_end] = study_preds_accumulator / max(total_weights, 1e-6)
        del results, batch_passes, batch_lats, batch_cflags, batch_metas, c_flags_tensor, meta_tensor
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print(f"🎉 Raw Multi-Fold Inference Completed for {N_test:,} studies.")



🚀 Streaming Inference over 3 Hidden Test Studies (5 Fold Models)...
   Pipeline: 21 Slices @ 224x224 | Slices/plane: 7 | Batch size: 8 | Slice-Jitter TTA: True | Anatomical Flip TTA: False


Inferring Batches:   0%|          | 0/1 [00:00<?, ?it/s]

🎉 Raw Multi-Fold Inference Completed for 3 studies.


In [8]:
import numpy as np
from scipy.stats import rankdata

# =========================================================================
# 8. Post-Processing & Percentile Rank Normalization
# =========================================================================

processed_preds = final_predictions.copy()

# ⚡ Only apply Rank Normalization when evaluating on real Hidden Test set (N > 10).
# For preview dry-run (N <= 10), keep raw continuous model probabilities for inspection.
if CFG.use_rank_norm and N_test > 10:
    print("📊 Applying Column-Wise Percentile Rank Normalization across Hidden Test...")
    for j in range(processed_preds.shape[1]):
        col_vals = processed_preds[:, j]
        # Scale ranks to (0, 1] per column
        ranks = rankdata(col_vals, method="average") / len(col_vals)
        processed_preds[:, j] = ranks
    print(f"✅ Successfully Rank-Normalized all {processed_preds.shape[1]} Target Pathology Columns.")
else:
    print(f"ℹ️ Preserving continuous raw model probabilities for inspection (N_test={N_test} <= 10).")

# Ensure strictly bounded in [0.0001, 0.9999]
processed_preds = np.clip(processed_preds, 1e-4, 1.0 - 1e-4)

# Populate submission DataFrame
for j, col_name in enumerate(TARGET_COLS):
    sub_df[col_name] = processed_preds[:, j]

print("✅ Post-processing complete.")

ℹ️ Preserving continuous raw model probabilities for inspection (N_test=3 <= 10).
✅ Post-processing complete.


In [9]:
import os
import numpy as np
import pandas as pd

# =========================================================================
# 9. Strict Submission Sanity Checks & CSV Export
# =========================================================================

out_submission_path = '/kaggle/working/submission.csv' if os.path.exists('/kaggle/working') else './submission.csv'

# ── 1. Check shape against sample_submission ────────────────────────────────
sample_sub_check = pd.read_csv(sub_path)
assert sub_df.shape == sample_sub_check.shape, f"❌ Shape mismatch: {sub_df.shape} vs {sample_sub_check.shape}"
assert list(sub_df.columns) == list(sample_sub_check.columns), "❌ Column mismatch!"
assert (sub_df['StudyInstanceUID'].values == sample_sub_check['StudyInstanceUID'].values).all(), "❌ UID mismatch!"

# ── 2. Check for NaN, Inf, or null values ───────────────────────────────────
nan_count = sub_df[TARGET_COLS].isna().sum().sum()
inf_count = np.isinf(sub_df[TARGET_COLS].values).sum()
assert nan_count == 0, f"❌ Found {nan_count} NaN values in submission!"
assert inf_count == 0, f"❌ Found {inf_count} Inf values in submission!"

# ── 3. Check values are within [0.0, 1.0] ────────────────────────────────────
min_val = sub_df[TARGET_COLS].values.min()
max_val = sub_df[TARGET_COLS].values.max()
assert min_val >= 0.0 and max_val <= 1.0, f"❌ Values out of bounds: min={min_val}, max={max_val}"

# ── 4. Save final verified submission.csv ───────────────────────────────────
sub_df.to_csv(out_submission_path, index=False)
print(f"\n💾 SUCCESS! Saved verified submission to: {out_submission_path}")
print(f"   Shape: {sub_df.shape} | Total Test Predictions: {len(sub_df):,}")
print(f"   File size: {os.path.getsize(out_submission_path)/1024:.2f} KB")
print(f"\n📊 Sample Predictions (First 5 Rows):")
try:
    display(sub_df.head(5))
except Exception:
    print(sub_df.head(5))



💾 SUCCESS! Saved verified submission to: /kaggle/working/submission.csv
   Shape: (3, 13) | Total Test Predictions: 3
   File size: 0.70 KB

📊 Sample Predictions (First 5 Rows):


,StudyInstanceUID,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10047035057544427318...,0.066533,0.110810,0.945763,0.059211,0.958607,0.668700,0.870369,0.516516,0.203641,0.988858,0.12573,0.040106
1,1.2.826.0.1.3680043.8.498.10062861783145312629...,0.426691,0.175043,0.701392,0.100230,0.830123,0.745583,0.985370,0.838857,0.879485,0.691318,0.79355,0.418446
2,1.2.826.0.1.3680043.8.498.10067514707072572280...,0.046691,0.249637,0.339614,0.139405,0.762269,0.673991,0.909067,0.477749,0.157435,0.150497,0.04735,0.100172
